# Applied Python: pandas, PySpark, scikit-learn

**DS Readiness Training**

| Section | Who |
|---|---|
| **1. pandas core** — Superstore | ALL |
| **2. pandas → PySpark** — Superstore | DE live · DS observes |
| **3. pandas → sklearn** — Telco | DS live · DE observes |
| **4. Async brief** | ALL |

## The two datasets

| | Superstore | Telco |
|---|---|---|
| Rows × cols | 9,800 × 18 | 7,043 × 21 |
| Grain | one **order line** | one **customer** |
| Used for | pandas + PySpark | sklearn |
| Target | derived (`late` flag) | `Churn` — 26.5% positive |

## Two things before we start

1. **Run cells top to bottom.** Before submitting anything, use **Runtime → Restart and run all**. If it only works when cells are run out of order, it's broken.
2. **Cells marked "Predict first"** — commit to an answer before running. That's the whole method.

## You already know most of this

You spent three days on SQL. pandas is the same six operations with different syntax:

| SQL | pandas |
|---|---|
| `SELECT col` | `df[["col"]]` |
| `WHERE` | boolean mask |
| `GROUP BY` + aggregate | `groupby().agg()` |
| `JOIN` | `merge(how=...)` |
| `ORDER BY` | `sort_values()` |
| `LIMIT` | `.head()` |
| `COUNT(DISTINCT x)` | `nunique()` |
| Window function | `groupby().transform()` |
| `UNION ALL` | `concat()` |


---

# 0. Setup and loading

In [90]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 400)
pd.set_option("display.max_columns", 40)

print("pandas", pd.__version__)
print("numpy ", np.__version__)

# VERSION NOTE: on pandas 2.x a text column shows as `object` in .info().
# On pandas 3.0+ it shows as `str`. Same thing — a column of text.
# "object dtype" is the phrase used in interviews.
IS_PANDAS_3 = int(pd.__version__.split(".")[0]) >= 3
print("text columns will display as:", "str (pandas 3.x)" if IS_PANDAS_3 else "object (pandas 2.x)")

pandas 2.2.3
numpy  2.1.3
text columns will display as: object (pandas 2.x)


In [91]:
# ---------------------------------------------------------------------------
# TRAINER: set BASE to your hosted folder, e.g.
# BASE = "https://cdn.jsdelivr.net/gh/<user>/<repo>@main/data"
# Leave it empty to load from the local directory or a Colab upload.
# ---------------------------------------------------------------------------
BASE = "https://cdn.jsdelivr.net/gh/sumanshekhar698/data-engineering@main/datasets/"

SUPERSTORE_FILE = "SuperStoresKaggle.csv"
TELCO_FILE      = "TelcoCustomerKaggle.csv"


def resolve(name: str) -> str:
    """Find the file: hosted URL first, then common local locations."""
    candidates = []
    if BASE:
        candidates.append(BASE.rstrip("/") + "/" + name)
    candidates += [name, f"data/{name}", f"/content/{name}",
                   f"/mnt/user-data/uploads/{name}"]
    print("candidates:", candidates)
    for c in candidates:
        try:
            pd.read_csv(c, nrows=1)
            return c
        except Exception:
            continue
    raise FileNotFoundError(
        f"Could not find {name}. Set BASE, or upload the file:\n"
        f"    from google.colab import files; files.upload()")


SUPERSTORE_PATH = resolve(SUPERSTORE_FILE)
TELCO_PATH      = resolve(TELCO_FILE)
print("superstore ->", SUPERSTORE_PATH)
print("telco      ->", TELCO_PATH)

candidates: ['https://cdn.jsdelivr.net/gh/sumanshekhar698/data-engineering@main/datasets/SuperStoresKaggle.csv', 'SuperStoresKaggle.csv', 'data/SuperStoresKaggle.csv', '/content/SuperStoresKaggle.csv', '/mnt/user-data/uploads/SuperStoresKaggle.csv']
candidates: ['https://cdn.jsdelivr.net/gh/sumanshekhar698/data-engineering@main/datasets/TelcoCustomerKaggle.csv', 'TelcoCustomerKaggle.csv', 'data/TelcoCustomerKaggle.csv', '/content/TelcoCustomerKaggle.csv', '/mnt/user-data/uploads/TelcoCustomerKaggle.csv']
superstore -> https://cdn.jsdelivr.net/gh/sumanshekhar698/data-engineering@main/datasets/SuperStoresKaggle.csv
telco      -> https://cdn.jsdelivr.net/gh/sumanshekhar698/data-engineering@main/datasets/TelcoCustomerKaggle.csv


---

# 1. pandas core (ALL · 40 min)

## 1.1 Load and look

Two commands before anything else: `.info()` and `.describe()`. They are the diagnostic step, and skipping them is how bad data reaches production.

In [92]:
df = pd.read_csv(SUPERSTORE_PATH)

print("shape:", df.shape)
df.head(10)

shape: (9800, 18)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680
5,6,CA-2015-115812,09/06/2015,14/06/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.8600
6,7,CA-2015-115812,09/06/2015,14/06/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.2800
7,8,CA-2015-115812,09/06/2015,14/06/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.1520
8,9,CA-2015-115812,09/06/2015,14/06/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by S...,18.5040
9,10,CA-2015-115812,09/06/2015,14/06/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9000


In [93]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   object 
 2   Order Date     9800 non-null   object 
 3   Ship Date      9800 non-null   object 
 4   Ship Mode      9800 non-null   object 
 5   Customer ID    9800 non-null   object 
 6   Customer Name  9800 non-null   object 
 7   Segment        9800 non-null   object 
 8   Country        9800 non-null   object 
 9   City           9800 non-null   object 
 10  State          9800 non-null   object 
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   object 
 13  Product ID     9800 non-null   object 
 14  Category       9800 non-null   object 
 15  Sub-Category   9800 non-null   object 
 16  Product Name   9800 non-null   object 
 17  Sales          9800 non-null   float64
dtypes: float

### Predict first

Look at that output and answer three questions before scrolling:

1. Which columns are stored as **text** but shouldn't be?
2. Which column has **missing values**, and how many?
3. Which column has the **wrong numeric type**?

### The three problems `.info()` just revealed

**Problem 1 — `Order Date` and `Ship Date` are text.** They look like dates, so people assume pandas parsed them. It did not. You cannot do arithmetic, filter by year, or resample until they're converted.

**Problem 2 — `Postal Code` has 11 nulls and is `float64`.** Two bugs in one column. A postal code is an **identifier**, not a quantity — storing it as a float destroys leading zeros and invites nonsensical arithmetic like `mean(Postal Code)`.

**Problem 3 — 15 columns are text.** Some are genuinely categorical (`Region`, `Segment`), which matters for memory and for modelling.

In [94]:
# .describe() on numeric columns only by default
df.describe()

,Row ID,Postal Code,Sales
count,9800.000000,9789.000000,9800.000000
mean,4900.500000,55273.322403,230.769059
std,2829.160653,32041.223413,626.651875
min,1.000000,1040.000000,0.444000
25%,2450.750000,23223.000000,17.248000
50%,4900.500000,58103.000000,54.490000
75%,7350.250000,90008.000000,210.605000
max,9800.000000,99301.000000,22638.480000


In [95]:
# include="all" also profiles the text columns — unique counts, top value, frequency
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Row ID,9800.0,NaN,NaN,NaN,4900.5,2829.160653,1.0,2450.75,4900.5,7350.25,9800.0
Order ID,9800,4922,CA-2018-100111,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order Date,9800,1230,05/09/2017,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Date,9800,1326,26/09/2018,34,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Mode,9800,4,Standard Class,5859,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer ID,9800,793,WB-21850,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer Name,9800,793,William Brown,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Segment,9800,3,Consumer,5101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,9800,1,United States,9800,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,9800,529,New York City,891,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [96]:
# Null audit — do this on every dataset, every time
nulls = df.isna().sum()
# nulls
print(nulls[nulls > 0])
print()
print("total nulls:", df.isna().sum().sum())
print("duplicate rows:", df.duplicated().sum())
print()
# Cardinality tells you what's an identifier vs a category
for c in ["Order ID", "Customer ID", "Product ID", "Category", "Sub-Category",
          "Region", "State", "City", "Segment", "Ship Mode"]:
    print(f"  {c:<14} {df[c].nunique():>6,} distinct")

Postal Code    11
dtype: int64

total nulls: 11
duplicate rows: 0

  Order ID        4,922 distinct
  Customer ID       793 distinct
  Product ID      1,861 distinct
  Category            3 distinct
  Sub-Category       17 distinct
  Region              4 distinct
  State              49 distinct
  City              529 distinct
  Segment             3 distinct
  Ship Mode           4 distinct


> ### The grain question — ask it every time
>
> **9,800 rows but only 4,922 distinct `Order ID`s.** So one row is *not* one order — it's one **product line within an order**, averaging 1.99 lines per order.
>
> This is the same fan-out you verified in SQL (326 orders → 2,996 lines in classicmodels). Everything downstream depends on getting it right: `count()` on an order-level column will be inflated.

In [97]:
print("rows                :", f"{len(df):,}")
print("distinct orders     :", f"{df['Order ID'].nunique():,}")
print("lines per order     :", round(len(df) / df['Order ID'].nunique(), 2))
print("distinct customers  :", f"{df['Customer ID'].nunique():,}")
print("distinct products   :", f"{df['Product ID'].nunique():,}")

rows                : 9,800
distinct orders     : 4,922
lines per order     : 1.99
distinct customers  : 793
distinct products   : 1,861


## 1.2 Fixing types — and the date trap

This is the most important five minutes of the session.

In [98]:
# Look at the raw values first. What format is this?
df["Order Date"].head(8).tolist()

['08/11/2017',
 '08/11/2017',
 '12/06/2017',
 '11/10/2016',
 '11/10/2016',
 '09/06/2015',
 '09/06/2015',
 '09/06/2015']

### Predict first

`08/11/2017` — is that **8 November** or **11 August**?

You cannot tell from one value. You have to test.

In [99]:
# Try it as MM/DD/YYYY (the US convention)
as_us = pd.to_datetime(df["Order Date"], format="%m/%d/%Y", errors="coerce")
#  errors="coerce" :: "If you can't convert a value into this date format, don't throw an error. Instead, convert it to NaT."
print("parsed as MM/DD/YYYY -> failures:", f"{as_us.isna().sum():,}")

# Try it as DD/MM/YYYY
as_intl = pd.to_datetime(df["Order Date"], format="%d/%m/%Y", errors="coerce")
print("parsed as DD/MM/YYYY -> failures:", f"{as_intl.isna().sum():,}")

parsed as MM/DD/YYYY -> failures: 5,841
parsed as DD/MM/YYYY -> failures: 0


> ### The trap, stated plainly
>
> **5,841 of 9,800 rows fail as MM/DD/YYYY. Zero fail as DD/MM/YYYY.** So the format is DD/MM/YYYY.
>
> But look at what that means for the 3,959 rows that *would* have parsed either way. `08/11/2017` parses under both formats — **as two different dates**. If you'd used `errors="coerce"` and only checked for nulls, you'd have concluded "only 60% failed, let me drop those" and silently kept 3,959 **wrong dates** with no error, no warning, nothing.
>
> **This is the worst class of data bug: silently wrong, not loudly broken.** Always state the format explicitly.

In [100]:
# Convert properly. ALWAYS pass an explicit format when you know it.
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%d/%m/%Y")
df["Ship Date"]  = pd.to_datetime(df["Ship Date"],  format="%d/%m/%Y")


print(df[["Order Date", "Ship Date"]].dtypes)
print()
print("date range:", df["Order Date"].min().date(), "to", df["Order Date"].max().date())

# Order Date
#     ↓
# datetime64[ns]
#     ↓
#    .dt
#     ↓
# ┌────────┬────────┬
# year    month    day
print("years present:", sorted(df["Order Date"].dt.year.unique()))

Order Date    datetime64[ns]
Ship Date     datetime64[ns]
dtype: object

date range: 2015-01-03 to 2018-12-30
years present: [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018)]


In [101]:
# Postal Code: an identifier, not a number
print("before:", df["Postal Code"].dtype)
print("sample:", df["Postal Code"].head(3).tolist())
print()

# Why float is wrong: leading zeros are already destroyed for low codes,
# and nonsense operations are permitted.
print("nonsense that pandas allows:", round(df["Postal Code"].mean(), 2))
print()

df["Postal Code"] = (df["Postal Code"]
                     .astype("Int64")            # nullable integer — keeps the 11 NaN
                     .astype("string")
                     .str.zfill(5))              # restore 5-digit form

# Postal Code
#      ↓
#   float64
#      ↓
# .astype("Int64")
#      ↓
# nullable integer
#      ↓
# .astype("string")
#      ↓
#     text
#      ↓
# .str.zfill(5)
#      ↓
# 5-character postal code

print("after :", df["Postal Code"].dtype)
print("sample:", df["Postal Code"].head(3).tolist())
print("nulls :", df["Postal Code"].isna().sum())

before: float64
sample: [42420.0, 42420.0, 90036.0]

nonsense that pandas allows: 55273.32

after : string
sample: ['42420', '42420', '90036']
nulls : 11


> **`Int64` (capital I) vs `int64`** — the capitalised version is pandas' *nullable* integer type. Plain `int64` cannot hold NaN, which is exactly why the column arrived as `float64` in the first place.

### Memory: the `category` dtype

Low-cardinality text columns are wasteful as strings. `category` stores each distinct value once plus small integer codes.

In [102]:
before = df.memory_usage(deep=True).sum()
# print(df.memory_usage())
print()
print(df.memory_usage().sum())
print()

for c in ["Ship Mode", "Segment", "Country", "Region", "Category", "Sub-Category"]:
    df[c] = df[c].astype("category")
    # catrgory is a data type in Pandas

after = df.memory_usage(deep=True).sum()
print(f"before: {before/1e6:>7.2f} MB")
print(f"after : {after/1e6:>7.2f} MB   ({(1-after/before)*100:.0f}% smaller)")
print()
print(df["Region"].dtype)
print("categories:", list(df["Region"].cat.categories))


1411332

before:    8.72 MB
after :    5.32 MB   (39% smaller)

category
categories: ['Central', 'East', 'South', 'West']


## 1.3 Selecting and filtering

The SQL equivalents you already know.

In [103]:
# SELECT — note the difference between one bracket and two
print(type(df["Sales"]))        # Series  — one column
print(type(df[["Sales"]]))      # DataFrame — a table with one column

df[["Order ID", "Order Date", "Category", "Sales"]].head()

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>


,Order ID,Order Date,Category,Sales
0,CA-2017-152156,2017-11-08,Furniture,261.9600
1,CA-2017-152156,2017-11-08,Furniture,731.9400
2,CA-2017-138688,2017-06-12,Office Supplies,14.6200
3,US-2016-108966,2016-10-11,Furniture,957.5775
4,US-2016-108966,2016-10-11,Office Supplies,22.3680


In [104]:
# WHERE — a boolean mask
mask = df["Sales"] > 5000
print("mask is a boolean Series:", mask.dtype, "| True count:", mask.sum())

df.loc[mask, ["Order ID", "Product Name", "Sales"]].head()

mask is a boolean Series: bool | True count: 19


,Order ID,Product Name,Sales
165,CA-2015-139892,Lexmark MX611dhe Monochrome Laser Printer,8159.952
509,CA-2016-145352,Fellowes PB500 Electric Punch Plastic Comb Bin...,6354.950
683,US-2018-168116,Cubify CubeX 3D Printer Triple Head Print,7999.980
2505,CA-2015-143917,High Speed Automatic Electric Letter Opener,8187.650
2623,CA-2018-127180,Canon imageCLASS 2200 Advanced Copier,11199.968


In [105]:
# Multiple conditions: & and | , and PARENTHESES ARE MANDATORY.
# & binds tighter than > in Python, so without brackets you get a TypeError.
high_west = df[(df["Sales"] > 1000) & (df["Region"] == "West")]
print("high-value West lines:", len(high_west))
print(high_west[["Order ID", "Product Name", "Sales"]].head())
print()
print(high_west[["Order ID", "Product Name", "Sales"]].head())
print()

# The SQL OR
tech_or_furn = df[(df["Category"] == "Technology") | (df["Category"] == "Furniture")]
print("Technology or Furniture:", len(tech_or_furn))

# Cleaner with isin() — the SQL IN
print("same via isin():", len(df[df["Category"].isin(["Technology", "Furniture"])]))

# NOT — use ~ , not `not`
print("everything else    :", len(df[~df["Category"].isin(["Technology", "Furniture"])]))

high-value West lines: 157
           Order ID                                       Product Name     Sales
10   CA-2015-115812           Chromcraft Rectangular Conference Tables  1706.184
24   CA-2016-106320      Bretford CR4500 Series Slim Rectangular Table  1044.630
67   CA-2015-106376  Hunt BOSTON Model 1606 High-Volume Electric Pe...  1113.024
251  CA-2017-145625                 Logitech P710e Mobile Speakerphone  3347.370
282  CA-2016-130890            Bevis Round Bullnose 29" High Table Top  1038.840

           Order ID                                       Product Name     Sales
10   CA-2015-115812           Chromcraft Rectangular Conference Tables  1706.184
24   CA-2016-106320      Bretford CR4500 Series Slim Rectangular Table  1044.630
67   CA-2015-106376  Hunt BOSTON Model 1606 High-Volume Electric Pe...  1113.024
251  CA-2017-145625                 Logitech P710e Mobile Speakerphone  3347.370
282  CA-2016-130890            Bevis Round Bullnose 29" High Table Top  1038.840


In [106]:
# BETWEEN
print("sales 100-500:", len(df[df["Sales"].between(100, 500)]))   # inclusive, like SQL

# LIKE
print("names containing 'Chair':", df["Product Name"].str.contains("Chair", case=False).sum())

# Date filtering — only possible now the column is a real datetime
print("orders in 2017:", len(df[df["Order Date"].dt.year == 2017]))
print("Q4 2017       :", len(df[(df["Order Date"] >= "2017-10-01") &
                                (df["Order Date"] <= "2017-12-31")]))

# .query() — SQL-ish string syntax. Readable for simple filters.
print("via query()    :", len(df.query("Sales > 1000 and Region == 'West'")))

sales 100-500: 2553
names containing 'Chair': 689
orders in 2017: 2534
Q4 2017       : 896
via query()    : 157


In [107]:
# .loc vs .iloc — LABEL vs POSITION
print("iloc[0, 1]  (position):", df.iloc[0, 1])
print("loc[0, 'Order ID'] (label):", df.loc[0, "Order ID"])
print()

# .loc slices are INCLUSIVE of the endpoint; .iloc is not. A classic off-by-one.
print("loc[0:2] rows :", len(df.loc[0:2]))      # 3 rows
print("iloc[0:2] rows:", len(df.iloc[0:2]))     # 2 rows

iloc[0, 1]  (position): CA-2017-152156
loc[0, 'Order ID'] (label): CA-2017-152156

loc[0:2] rows : 3
iloc[0:2] rows: 2


## 1.4 Derived columns

Now that the dates are real, we can compute the thing the raw data doesn't contain: **how long shipping took**.

In [108]:
# Date arithmetic gives a timedelta; .dt.days makes it a number
df["ship_delay"] = (df["Ship Date"] - df["Order Date"]).dt.days

# print(df["ship_delay"].describe())
# print()

print(df.head())
print()


# print(df["ship_delay"].head())
# print()

print("delay distribution:")
print(df["ship_delay"].value_counts().sort_index())

   Row ID        Order ID Order Date  Ship Date       Ship Mode Customer ID    Customer Name    Segment        Country             City       State Postal Code Region       Product ID         Category Sub-Category                                       Product Name     Sales  ship_delay
0       1  CA-2017-152156 2017-11-08 2017-11-11    Second Class    CG-12520      Claire Gute   Consumer  United States        Henderson    Kentucky       42420  South  FUR-BO-10001798        Furniture    Bookcases                  Bush Somerset Collection Bookcase  261.9600           3
1       2  CA-2017-152156 2017-11-08 2017-11-11    Second Class    CG-12520      Claire Gute   Consumer  United States        Henderson    Kentucky       42420  South  FUR-CH-10000454        Furniture       Chairs  Hon Deluxe Fabric Upholstered Stacking Chairs,...  731.9400           3
2       3  CA-2017-138688 2017-06-12 2017-06-16    Second Class    DV-13045  Darrin Van Huff  Corporate  United States      Los Angeles  Ca

In [109]:
# A binary flag — this is our target for the modelling discussion later
df["late"] = (df["ship_delay"] > 5).astype(int)

print("late rate:", f"{df['late'].mean():.1%}")
print(df["late"].value_counts())
print()

# np.where for two outcomes
df["value_band"] = np.where(df["Sales"] > 500, "high", "low")
print(df["value_band"].value_counts().to_dict())

# pd.cut for multiple bands — the equivalent of a CASE expression
df["sales_band"] = pd.cut(df["Sales"],
                          bins=[0, 50, 200, 1000, np.inf],
                          labels=["micro", "small", "medium", "large"])
print()
print(df["sales_band"].value_counts().sort_index())

late rate: 18.2%
late
0    8015
1    1785
Name: count, dtype: int64

{'low': 8654, 'high': 1146}

sales_band
micro     4757
small     2506
medium    2075
large      462
Name: count, dtype: int64


In [110]:
# Calendar features from a datetime — one line each
df["year"]      = df["Order Date"].dt.year
df["month"]     = df["Order Date"].dt.month
df["quarter"]   = df["Order Date"].dt.quarter
df["weekday"]   = df["Order Date"].dt.day_name()
df["year_month"] = df["Order Date"].dt.to_period("M").astype(str)

df[["Order Date", "year", "quarter", "weekday", "year_month"]].head()

,Order Date,year,quarter,weekday,year_month
0,2017-11-08,2017,4,Wednesday,2017-11
1,2017-11-08,2017,4,Wednesday,2017-11
2,2017-06-12,2017,2,Monday,2017-06
3,2016-10-11,2016,4,Tuesday,2016-10
4,2016-10-11,2016,4,Tuesday,2016-10


In [111]:
# .assign() adds columns without mutating — chainable, and the modern style
summary = (df
           .assign(revenue_per_day=lambda d: d["Sales"] / d["ship_delay"].clip(lower=1))
           .loc[:, ["Order ID", "Sales", "ship_delay", "revenue_per_day"]]
           .head())
summary

,Order ID,Sales,ship_delay,revenue_per_day
0,CA-2017-152156,261.9600,3,87.320000
1,CA-2017-152156,731.9400,3,243.980000
2,CA-2017-138688,14.6200,4,3.655000
3,US-2016-108966,957.5775,7,136.796786
4,US-2016-108966,22.3680,7,3.195429


## 1.5 Cleaning: nulls, duplicates, categories

**The question to ask before filling any null: what does missing MEAN here?** One blanket `fillna(0)` is almost always wrong.

In [112]:
# Which rows have the missing postal codes, and does it matter?
missing_pc = df[df["Postal Code"].isna()]
print("rows with no postal code:", len(missing_pc))
print(missing_pc[["City", "State", "Region"]].drop_duplicates())

rows with no postal code: 11
            City    State Region
2234  Burlington  Vermont   East


> All 11 come from one city. So the null is a **data-entry gap**, not a meaningful absence — and it's recoverable from `City`/`State` if you had a lookup. That's a very different decision from Telco's blank `TotalCharges` (section 3), where the blank *means* something.

In [113]:
# The three options, and when each is right
print("dropna(subset=) — drop rows:", len(df.dropna(subset=["Postal Code"])), "rows remain")
print("fillna('UNKNOWN') — flag it:", df["Postal Code"].fillna("UNKNOWN").eq("UNKNOWN").sum(), "flagged")

# df = df.dropna(subset=["Postal Code"])
# df.dropna(subset=["Postal Code"], inplace=True)
print()
print("Here: keep the rows. 11 missing postal codes don't invalidate 11 sales figures.")
df["Postal Code"] = df["Postal Code"].fillna("UNKNOWN")
print("nulls remaining in the whole frame:", df.isna().sum().sum())

dropna(subset=) — drop rows: 9789 rows remain
fillna('UNKNOWN') — flag it: 11 flagged

Here: keep the rows. 11 missing postal codes don't invalidate 11 sales figures.
nulls remaining in the whole frame: 0


In [114]:
# Duplicates. This dataset has none at the row level — but Row ID is a
# surrogate key, so "no duplicate rows" is guaranteed and therefore meaningless.
print("full-row duplicates      :", df.duplicated().sum())

# The useful question: duplicates on the BUSINESS key
biz_key = ["Order ID", "Product ID"]
print("duplicates on (Order ID, Product ID):", df.duplicated(subset=biz_key).sum())
# df.duplicated(subset=biz_key, keep="first")
# df.duplicated(subset=biz_key, keep="false")

print()
print("keep='first' / 'last' / False control WHICH copies you keep.")
print("Always name the subset. drop_duplicates() with no subset only removes")
print("rows identical in EVERY column, which a surrogate key prevents.")

full-row duplicates      : 0
duplicates on (Order ID, Product ID): 8

keep='first' / 'last' / False control WHICH copies you keep.
Always name the subset. drop_duplicates() with no subset only removes
rows identical in EVERY column, which a surrogate key prevents.


In [115]:
# Categorical hygiene — check before you trust a groupby
print("Region      :", sorted(df["Region"].unique()))
print("Segment     :", sorted(df["Segment"].unique()))
print("Ship Mode   :", sorted(df["Ship Mode"].unique()))
print("Category    :", sorted(df["Category"].unique()))
print()
print("Clean here. But run this defensively on any new dataset — a stray")
print("' West' or 'west' silently becomes a separate group with no error.")
print()
# Defensive normalisation, for text columns you haven't verified
sample = pd.Series([" West", "west", "WEST", "West "])
print("raw distinct     :", sample.nunique())
print("after strip/title:", sample.str.strip().str.title().nunique())

Region      : ['Central', 'East', 'South', 'West']
Segment     : ['Consumer', 'Corporate', 'Home Office']
Ship Mode   : ['First Class', 'Same Day', 'Second Class', 'Standard Class']
Category    : ['Furniture', 'Office Supplies', 'Technology']

Clean here. But run this defensively on any new dataset — a stray
' West' or 'west' silently becomes a separate group with no error.

raw distinct     : 4
after strip/title: 1


## 1.6 Aggregation — `groupby`

Identical semantics to SQL's `GROUP BY`: split into groups, apply an aggregate, combine.

In [116]:
# The simplest form
df.groupby("Region", observed=True)["Sales"].sum().sort_values(ascending=False)
# observed=True → show categories actually observed in the data

,Sales
Region,
West,710219.6845
East,669518.7260
Central,492646.9132
South,389151.4590


In [117]:
# Multiple aggregations with named output — the readable modern form.
# This is  SELECT Region, SUM(Sales), COUNT(DISTINCT Order ID), AVG(Sales) ...
region = (df.groupby("Region", observed=True)
            .agg(total_sales=("Sales", "sum"),
                 orders=("Order ID", "nunique"),       # <<< COUNT(DISTINCT ...)
                 lines=("Order ID", "count"),          # <<< COUNT(*) — inflated by fan-out
                 avg_line=("Sales", "mean"),
                 customers=("Customer ID", "nunique"),
                 late_rate=("late", "mean"))
            .round(2)
            .sort_values("total_sales", ascending=False))
region

,total_sales,orders,lines,avg_line,customers,late_rate
Region,,,,,,
West,710219.68,1587,3140,226.18,681,0.19
East,669518.73,1369,2785,240.40,669,0.17
Central,492646.91,1156,2277,216.36,626,0.19
South,389151.46,810,1598,243.52,509,0.18


> ### `count` vs `nunique` — the fan-out lesson, again
>
> `lines` and `orders` differ because we're at **line grain**. `count` counts rows; `nunique` counts distinct orders. This is the exact `COUNT(*)` vs `COUNT(DISTINCT ...)` distinction from Day 1, and it is the single most common analytics bug.

In [118]:
# Two-level grouping — the Category > Sub-Category hierarchy
cat = (df.groupby(["Category", "Sub-Category"], observed=True)
         .agg(sales=("Sales", "sum"), lines=("Sales", "size"))
         .round(2)
         .sort_values(["Category", "sales"], ascending=[True, False]))
print(cat.head(30))
print()
print("shape:", cat.shape, "--- 17 Category/Sub-Category combinations across 3 categories")

                                  sales  lines
Category        Sub-Category                  
Furniture       Chairs        322822.73    607
                Tables        202810.63    314
                Bookcases     113813.20    226
                Furnishings    89212.02    931
Office Supplies Storage       219343.39    832
                Binders       200028.78   1492
                Appliances    104618.40    459
                Paper          76828.30   1338
                Supplies       46420.31    184
                Art            26705.41    785
                Envelopes      16128.05    248
                Labels         12347.73    357
                Fasteners       3001.96    214
Technology      Phones        327782.45    876
                Machines      189238.63    115
                Accessories   164186.70    756
                Copiers       146248.09     66

shape: (17, 2) --- 17 Category/Sub-Category combinations across 3 categories


In [119]:
# HAVING — filter groups AFTER aggregating
state = (df.groupby("State", observed=True)
           .agg(sales=("Sales", "sum"), orders=("Order ID", "nunique"))
           .round(2))

print("all states      :", len(state))
print("with >20 orders :", len(state[state["orders"] > 20]))     # <<< this is HAVING
print()
print(state[state["orders"] > 20].sort_values("sales", ascending=False).head(80))

all states      : 49
with >20 orders : 35

                    sales  orders
State                            
California      446306.46    1002
New York        306361.15     547
Texas           168572.53     480
Washington      135206.85     254
Pennsylvania    116276.65     285
Florida          88436.53     197
Illinois         79236.52     270
Michigan         76136.07     116
Ohio             75130.35     227
Virginia         70636.72     115
North Carolina   55165.96     134
Indiana          48718.40      70
Georgia          48219.11      87
Kentucky         36458.39      59
Arizona          35272.66     107
New Jersey       34610.97      58
Colorado         31841.60      78
Wisconsin        31173.43      52
Tennessee        30661.87      91
Minnesota        29863.15      44
Massachusetts    28634.43      62
Delaware         27323.00      43
Maryland         23705.52      46
Rhode Island     22525.03      24
Missouri         22205.15      30
Oklahoma         19683.39      34
Alaba

In [120]:
# .transform() — a group-level value on EVERY row, without collapsing.


# This is pandas' window function.
# df["region_total"] = df.groupby("Region", observed=True)["Sales"].transform("sum")

# df["pct_of_region"] = (df["Sales"] / df["region_total"] * 100).round(3)

# Rank within group — the ROW_NUMBER() OVER (PARTITION BY ...) equivalent
# df["rank_in_category"] = (df.groupby("Category", observed=True)["Sales"]
#                             .rank(method="dense", ascending=False))

# df.loc[df["rank_in_category"] <= 2,
#        ["Category", "Product Name", "Sales", "rank_in_category"]].sort_values(
#     ["Category", "rank_in_category"]).head(6)




# 1. Add region_total
df["region_total"] = df.groupby("Region", observed=True)["Sales"].transform("sum")

print("After adding region_total:")
print(df[["Region", "Sales", "region_total"]].head(20))
print()


# 2. Add pct_of_region
df["pct_of_region"] = (
    df["Sales"] / df["region_total"] * 100
).round(3)

print("After adding pct_of_region:")
print(df[["Region", "Sales", "region_total", "pct_of_region"]].head())
print()


# 3. Add rank_in_category
df["rank_in_category"] = (
    df.groupby("Category", observed=True)["Sales"]
      .rank(method="dense", ascending=False)
)

print("After adding rank_in_category:")
print(df[[
    "Category",
    "Product Name",
    "Sales",
    "rank_in_category",
    "region_total"
]].head())
print()


# 4. Filter top 2 per category and sort
result = (
    df.loc[
        df["rank_in_category"] <= 2,
        ["Category", "Product Name", "Sales", "rank_in_category"]
    ]
    .sort_values(["Category", "rank_in_category"])
    .head(6)
)

print("Final result — top 2:")
print(result)

After adding region_total:
     Region      Sales  region_total
0     South   261.9600   389151.4590
1     South   731.9400   389151.4590
2      West    14.6200   710219.6845
3     South   957.5775   389151.4590
4     South    22.3680   389151.4590
5      West    48.8600   710219.6845
6      West     7.2800   710219.6845
7      West   907.1520   710219.6845
8      West    18.5040   710219.6845
9      West   114.9000   710219.6845
10     West  1706.1840   710219.6845
11     West   911.4240   710219.6845
12    South    15.5520   389151.4590
13     West   407.9760   710219.6845
14  Central    68.8100   492646.9132
15  Central     2.5440   492646.9132
16  Central   665.8800   492646.9132
17     West    55.5000   710219.6845
18     West     8.5600   710219.6845
19     West   213.4800   710219.6845

After adding pct_of_region:
  Region     Sales  region_total  pct_of_region
0  South  261.9600   389151.4590          0.067
1  South  731.9400   389151.4590          0.188
2   West   14.6200   71

> `rank(method="dense")` is `DENSE_RANK()`. `method="first"` is `ROW_NUMBER()`. `method="min"` is `RANK()`. Same three functions, same tie behaviour as Day 2.

In [121]:
# pivot_table — cross-tabulation. There's no PIVOT keyword in MySQL;
# in pandas it's one call.
pivot = pd.pivot_table(df,
                       index="Region",
                       columns="Category",
                       values="Sales",
                       aggfunc="sum",         # mean, count
                       margins=True,          # adds the row/column totals as 'All'
                       observed=True).round(0)
pivot

Category,Furniture,Office Supplies,Technology,All
Region,,,,
Central,160317.0,163590.0,168739.0,492647.0
East,206461.0,199941.0,263117.0,669519.0
South,116531.0,124425.0,148195.0,389151.0
West,245348.0,217467.0,247405.0,710220.0
All,728659.0,705422.0,827456.0,2261537.0


In [122]:
# Time series aggregation — resample needs a DatetimeIndex
monthly = (df.set_index("Order Date")
             .resample("ME")["Sales"]         # "ME" = month end ::
            # In newer Pandas versions, "ME" explicitly means month end. Older code commonly used "M" for this.
             .sum()
             .round(2))

print(monthly.head(6))
print()
print("months:", len(monthly))
print()
# Month-over-month change — the LAG() equivalent
mom = monthly.pct_change().mul(100).round(1)

# mom = (monthly.pct_change() * 100).round(1)
print("MoM % change (first 6):")
print(mom.head(6))

Order Date
2015-01-31    14205.71
2015-02-28     4519.89
2015-03-31    55205.80
2015-04-30    27906.86
2015-05-31    23644.30
2015-06-30    34322.94
Freq: ME, Name: Sales, dtype: float64

months: 48

MoM % change (first 6):
Order Date
2015-01-31       NaN
2015-02-28     -68.2
2015-03-31    1121.4
2015-04-30     -49.4
2015-05-31     -15.3
2015-06-30      45.2
Freq: ME, Name: Sales, dtype: float64


## 1.7 Merge — built via a normalisation exercise

Superstore is **one flat, fully denormalised table**. That's the shape you were taught to avoid.

So: split it into proper tables, then merge them back. Two lessons for the price of one.

In [123]:
# Split into 3NF-ish tables. drop_duplicates is doing real work here.
customers = (df[["Customer ID", "Customer Name", "Segment"]]
             .drop_duplicates("Customer ID")
             .reset_index(drop=True))
# reset_index() creates a fresh index:

products = (df[["Product ID", "Category", "Sub-Category", "Product Name"]]
            .drop_duplicates("Product ID")
            .reset_index(drop=True))

orders = (df[["Order ID", "Order Date", "Ship Date", "Ship Mode", "Customer ID"]]
          .drop_duplicates("Order ID")
          .reset_index(drop=True))

# we dont drop duplicates Because these rows are supposed to be different.
lines = df[["Row ID", "Order ID", "Product ID", "Sales"]].copy()

for name, t in [("customers", customers), ("products", products),
                ("orders", orders), ("lines", lines)]:
    print(f"{name:<10} {t.shape}")

customers  (793, 3)
products   (1861, 4)
orders     (4922, 5)
lines      (9800, 4)


### Predict first

The split above assumed `Customer ID → Segment` and `Product ID → Product Name` are **functional dependencies** — one customer has one segment, one product ID has one name.

Is that true in this data?

In [124]:
# Check the assumption. This is a real 3NF test on real data.
per_customer_city    = df.groupby("Customer ID", observed=True)["City"].nunique()
per_customer_segment = df.groupby("Customer ID", observed=True)["Segment"].nunique()
per_product_name     = df.groupby("Product ID", observed=True)["Product Name"].nunique()

print("customers with >1 City        :", (per_customer_city > 1).sum(), f"of {len(per_customer_city)}")
# customers = df[["Customer ID", "Customer Name", "Segment", "City"]].drop_duplicates("Customer ID")
# Cannot Decide as we dont know which city will go in

print("customers with >1 Segment     :", (per_customer_segment > 1).sum()) #It means every Customer ID has exactly one Segment.

print("Product IDs with >1 Product Name:", (per_product_name > 1).sum())
# products = (df[["Product ID", "Category", "Sub-Category", "Product Name"]].drop_duplicates("Product ID"))
# Cannot Decide as we dont know which Product Name to retailn

customers with >1 City        : 779 of 793
customers with >1 Segment     : 0
Product IDs with >1 Product Name: 32


> ### What that tells you
>
> **`Customer ID → Segment` holds.** Zero violations, so `Segment` legitimately belongs in the `customers` table.
>
> **`Customer ID → City` does NOT hold.** 779 of 793 customers appear with more than one city — because a customer *ships to different addresses over time*. City is a fact about the **order**, not the customer. Putting it in `customers` would silently discard information.
>
> **`Product ID → Product Name` is violated 32 times.** That is not a modelling subtlety, it's a **data quality defect** — the same product ID carrying two different names. In a real pipeline this is a validation-gate failure, and it's exactly the Day 5 topic.
>
> This is why normalisation is a data-modelling skill and not a mechanical rule. The data tells you where the dependencies actually are.

In [125]:
# Inspect one of the violating products
bad_products = per_product_name[per_product_name > 1].index[:2]
df[df["Product ID"].isin(bad_products)][["Product ID", "Product Name"]].drop_duplicates()

,Product ID,Product Name
66,FUR-CH-10001146,"Global Value Mid-Back Manager's Chair, Gray"
128,FUR-CH-10001146,"Global Task Chair, Black"
2115,FUR-BO-10002213,DMI Eclipse Executive Suite Bookcases
2471,FUR-BO-10002213,"Sauder Forest Hills Library, Woodland Oak Finish"


In [126]:
# Now merge back. how= behaves EXACTLY like SQL's join types.
rebuilt = (lines
           .merge(orders,    on="Order ID",    how="left")
           .merge(customers, on="Customer ID", how="left")
           .merge(products,  on="Product ID",  how="left"))

print("original rows:", f"{len(df):,}")
print("rebuilt rows :", f"{len(rebuilt):,}")
print("sales match  :", round(df["Sales"].sum(), 2) == round(rebuilt["Sales"].sum(), 2))
rebuilt.head(3)

original rows: 9,800
rebuilt rows : 9,800
sales match  : True


,Row ID,Order ID,Product ID,Sales,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Category,Sub-Category,Product Name
0,1,CA-2017-152156,FUR-BO-10001798,261.96,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,Furniture,Bookcases,Bush Somerset Collection Bookcase
1,2,CA-2017-152156,FUR-CH-10000454,731.94,2017-11-08,2017-11-11,Second Class,CG-12520,Claire Gute,Consumer,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,..."
2,3,CA-2017-138688,OFF-LA-10000240,14.62,2017-06-12,2017-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...


In [127]:
# validate= catches unexpected fan-out AT MERGE TIME instead of in a report
try:
    lines.merge(orders, on="Order ID", how="left", validate="many_to_one")
    print("many_to_one: OK — each line maps to exactly one order")
except Exception as e:
    print("validate failed:", e)

try:
    orders.merge(lines, on="Order ID", how="left", validate="one_to_one")
except Exception as e:
    print("one_to_one   : correctly rejected —", str(e)[:70])
print()
print("Options: one_to_one, one_to_many, many_to_one, many_to_many")
print("Use it on every production merge. It turns a silent wrong number")
print("into a loud failure.")

many_to_one: OK — each line maps to exactly one order
one_to_one   : correctly rejected — Merge keys are not unique in right dataset; not a one-to-one merge

Options: one_to_one, one_to_many, many_to_one, many_to_many
Use it on every production merge. It turns a silent wrong number
into a loud failure.


In [128]:
# Demonstrating the join types. Inject an orphan key first, since this
# dataset is internally consistent.
orphan_lines = pd.concat([
    lines.head(5),
    pd.DataFrame({"Row ID": [99999], "Order ID": ["XX-9999-999999"],
                  "Product ID": ["NOPE-000"], "Sales": [123.45]})
], ignore_index=True)

inner = orphan_lines.merge(orders, on="Order ID", how="inner")
left  = orphan_lines.merge(orders, on="Order ID", how="left")

print("input lines :", len(orphan_lines))
print("inner join  :", len(inner), " <<< the orphan is SILENTLY DROPPED")
print("left join   :", len(left),  " <<< the orphan survives with NaN")
print()
# The anti-join: LEFT JOIN ... WHERE right IS NULL
print("orphans found via left + isna():")
print(left[left["Customer ID"].isna()][["Order ID", "Product ID", "Sales"]])

input lines : 6
inner join  : 5  <<< the orphan is SILENTLY DROPPED
left join   : 6  <<< the orphan survives with NaN

orphans found via left + isna():
         Order ID Product ID   Sales
5  XX-9999-999999   NOPE-000  123.45


In [129]:
# indicator=True labels the source of every row — great for diagnosis

# how="outer" → keep everything from both tables
# indicator=True → add a special _merge column telling you where each row came from

        #       orphan_lines        orders
        #       (LEFT)              (RIGHT)

        #             ┌───────────────┐
        #             │               │
        #             │    BOTH       │
        #             │      5        │
        #             │               │
        #             └───────────────┘

        # LEFT ONLY                  RIGHT ONLY
        # 1 orphan                   4919 orders
        # XX-9999                    no matching line

checked = orphan_lines.merge(orders, on="Order ID", how="outer", indicator=True)
print(checked["_merge"].value_counts())
print()
print("left_only  = in lines but no matching order (orphan FK)")
print("right_only = order with no lines")
print("both       = matched")

_merge
right_only    4919
both             5
left_only        1
Name: count, dtype: int64

left_only  = in lines but no matching order (orphan FK)
right_only = order with no lines
both       = matched


In [130]:
# concat — the UNION ALL equivalent (stacks rows instead of adding columns)
first_half  = df[df["year"] <= 2016]
second_half = df[df["year"] >= 2017]

stacked = pd.concat([first_half, second_half], ignore_index=True)
# ignore_index=True :: pandas creates a fresh sequential index:
print(f"{len(first_half):,} + {len(second_half):,} = {len(stacked):,}  (original {len(df):,})")




# first_half
# A
# B
# C

#         +

# second_half
# D
# E
# F

#         ↓

# stacked
# A
# B
# C
# D
# E
# F

4,008 + 5,792 = 9,800  (original 9,800)


## 1.8 Vectorisation — the 40-second loop

**The scenario:** someone writes a loop over a DataFrame and it takes forever. Here's why, and the fix.

In [131]:
import time

# Four ways to compute the same thing, worst to best
n = len(df)

# 1. iterrows — the worst. Builds a Series object for every single row.
t0 = time.perf_counter()
out = []
# The first value is the row's index, which you're not using, so you call it _.
# Row 1 → Series → get Sales → multiply
# Row 2 → Series → get Sales → multiply
# Row 3 → Series → get Sales → multiply
# ...
for _, row in df.iterrows():
    out.append(row["Sales"] * 1.18)
t_iterrows = time.perf_counter() - t0

# 2. apply(axis=1) — better, still a Python function call per row
# row 1 → Python function
# row 2 → Python function
# row 3 → Python function
# ...
t0 = time.perf_counter()


# axis=1  → DOWN the rows
# axis=0  → ACROSS the columns
_ = df.apply(lambda r: r["Sales"] * 1.18, axis=1)
t_apply_row = time.perf_counter() - t0

# 3. apply on a Series — one call per value, no row objects
t0 = time.perf_counter()
# Sales value → function
_ = df["Sales"].apply(lambda x: x * 1.18)
t_apply_col = time.perf_counter() - t0

# 4. Vectorised — the whole operation runs in C, once
t0 = time.perf_counter()
_ = df["Sales"] * 1.18
# Sales column
#      ↓
# [100, 200, 300, ...]
#      ↓
# multiply entire array by 1.18
#      ↓
# [118, 236, 354, ...]
t_vector = time.perf_counter() - t0

print(f"{'iterrows':<18} {t_iterrows:>9.5f}s   {t_iterrows/t_vector:>8,.0f}x slower")
print(f"{'apply(axis=1)':<18} {t_apply_row:>9.5f}s   {t_apply_row/t_vector:>8,.0f}x slower")
print(f"{'Series.apply':<18} {t_apply_col:>9.5f}s   {t_apply_col/t_vector:>8,.0f}x slower")
print(f"{'vectorised':<18} {t_vector:>9.5f}s   baseline")
print()
print(f"On {n:,} rows. Scale to 10 million and iterrows becomes hours.")

iterrows             0.47049s      1,298x slower
apply(axis=1)        0.07853s        217x slower
Series.apply         0.00283s          8x slower
vectorised           0.00036s   baseline

On 9,800 rows. Scale to 10 million and iterrows becomes hours.


> ### The rule
>
> **If you are writing a `for` loop over a DataFrame, stop.** There is almost always a vectorised form.
>
> `iterrows()` is legitimate in roughly one case: when you genuinely need row-by-row sequential logic that depends on prior results, and even then `itertuples()` is faster.
>
> `apply(axis=1)` is the compromise people reach for. It's still 10–100× slower than vectorised, because it calls a Python function per row.

In [132]:
# Vectorised conditional logic — no loop needed
# np.where for two branches
df["tax"] = np.where(df["Category"] == "Technology", df["Sales"] * 0.18, df["Sales"] * 0.12)

# np.select for many branches — the CASE WHEN equivalent
conditions = [
    df["Sales"] > 5000,
    df["Sales"] > 1000,
    df["Sales"] > 100,
]
choices = ["enterprise", "large", "standard"]
df["deal_size"] = np.select(conditions, choices, default="small")

print(df["deal_size"].value_counts())
print()
print(df[["Sales", "Category", "tax", "deal_size"]].head())

deal_size
small         6104
standard      3234
large          443
enterprise      19
Name: count, dtype: int64

      Sales         Category        tax deal_size
0  261.9600        Furniture   31.43520  standard
1  731.9400        Furniture   87.83280  standard
2   14.6200  Office Supplies    1.75440     small
3  957.5775        Furniture  114.90930  standard
4   22.3680  Office Supplies    2.68416     small


## 1.9 Writing output

In [133]:
# CSV — universal, but untyped and uncompressed
region.to_csv("region_summary.csv")
df.to_csv("superstore_clean.csv", index=False)      # index=False, almost always


# CSV
#  ↓
# plain text
#  ↓
# large + doesn't preserve pandas dtypes reliably


# Parquet
#  ↓
# columnar binary format
#  ↓
# compressed + preserves schema/dtypes


# Parquet — columnar, compressed, PRESERVES DTYPES


# DataFrame
#    ↓
# write to file
#    ↓
# Parquet
#    ↓
# read file
#    ↓
# DataFrame again

# Typically pandas needs something such as PyArrow to read/write Parquet.

# Use CSV when you need maximum portability or human readability.
# Use Parquet when you're building a data/analytics pipeline and care about size, speed, and preserving the data schema.

try:
    df.to_parquet("superstore_clean.parquet", index=False)
    import os
    csv_mb = os.path.getsize("superstore_clean.csv") / 1e6
    pq_mb  = os.path.getsize("superstore_clean.parquet") / 1e6
    print(f"CSV    : {csv_mb:>6.2f} MB")
    print(f"Parquet: {pq_mb:>6.2f} MB   ({csv_mb/pq_mb:.1f}x smaller)")
    print()
    rt = pd.read_parquet("superstore_clean.parquet")
    print("dtypes survive the round trip:", rt["Order Date"].dtype, "|", rt["Region"].dtype)
except ImportError:
    print("pyarrow not installed — pip install pyarrow")
    print("Parquet keeps dtypes; CSV loses them. That's why pipelines use Parquet.")

CSV    :   2.94 MB
Parquet:   0.53 MB   (5.5x smaller)

dtypes survive the round trip: datetime64[ns] | category


> **Why every data pipeline you'll meet uses Parquet, not CSV:** smaller, columnar (so reading 2 of 20 columns reads only 2), and it stores the schema — so `Order Date` comes back as a datetime instead of text. Every CSV round trip re-creates the exact type problem we spent section 1.2 fixing.

## 1.10 Section 1 recap

| SQL | pandas | Used above |
|---|---|---|
| `SELECT` | `df[[...]]` | 1.3 |
| `WHERE` | boolean mask | 1.3 |
| `IN` / `BETWEEN` / `LIKE` | `.isin()` / `.between()` / `.str.contains()` | 1.3 |
| `GROUP BY` + aggregates | `.groupby().agg()` | 1.6 |
| `HAVING` | filter the aggregated frame | 1.6 |
| `COUNT(DISTINCT x)` | `nunique()` | 1.6 |
| Window function | `.transform()` / `.rank()` | 1.6 |
| `PIVOT` (not in MySQL) | `pivot_table()` | 1.6 |
| `JOIN` | `.merge(how=)` | 1.7 |
| `UNION ALL` | `.concat()` | 1.7 |
| `ORDER BY` / `LIMIT` | `.sort_values()` / `.head()` | throughout |

**The five things to remember**

1. `.info()` first, every time. It found all three defects in this dataset.
2. **State the date format explicitly.** Inferred parsing fails silently and wrongly.
3. Identifiers are strings, not numbers.
4. Ask what a null **means** before filling it.
5. `count` vs `nunique` after a merge — the fan-out never goes away.

---

# 2. pandas → PySpark (DE live · DS observes · 25 min)

**DS students:** you're observing, but don't switch off — "have you worked with distributed processing?" is a standard interview question, and knowing *why* Spark differs from pandas is a good 60-second answer.

**DE students:** this session is deliberately **not** Spark architecture. That's Day 6. Today is: the same six operations, different API, and the four places your pandas mental model breaks.

In [134]:
# Colab needs this once. Takes 1-2 minutes.
# Uncomment and run, then continue.
# !pip install -q pyspark

import importlib.util
HAS_SPARK = importlib.util.find_spec("pyspark") is not None
print("pyspark available:", HAS_SPARK)
if not HAS_SPARK:
    print()
    print("Run this in a cell, then re-run from here:")
    print("    !pip install -q pyspark")
    print()
    print("The cells below will skip cleanly until it's installed, so you can")
    print("still read the code and the notes.")

pyspark available: True


In [135]:
if HAS_SPARK:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.types import (StructType, StructField, StringType,
                                   DoubleType, IntegerType)

    spark = (SparkSession.builder
             .appName("Day4")
             .master("local[*]")              # all local cores
             .config("spark.sql.shuffle.partitions", "8")   # default 200 is silly locally
            #  And this is why partition count matters for Spark performance: too few partitions → poor parallelism; too many → scheduling/shuffle overhead.
             .getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark", spark.version)
    print("cores:", spark.sparkContext.defaultParallelism)
else:
    print("skipped — install pyspark first")

Spark 4.0.4
cores: 2


## 2.1 Reading, and why `inferSchema` is a trap

In [136]:
# ---------------------------------------------------------------------------
# Spark cannot read https:// — Hadoop FileSystem has no working HTTP driver.
# pandas can (it uses urllib). So: download once, hand Spark a local path.
#
# This is a real difference worth knowing: pandas reads URLs, Spark reads
# filesystems. In production the path would be abfss:// or s3a://, which Spark
# reads natively — HTTP is the odd one out.
# ---------------------------------------------------------------------------
import os, urllib.request

SPARK_DIR = "/content/spark_data" if os.path.exists("/content") else "spark_data"
os.makedirs(SPARK_DIR, exist_ok=True)


def localise(path: str, name: str) -> str:
    """Return a file:// URI Spark can read, downloading first if needed."""
    if str(path).startswith(("http://", "https://")):
        local = os.path.join(SPARK_DIR, name)
        if not os.path.exists(local):
            print(f"downloading {name} ...", end=" ")
            urllib.request.urlretrieve(path, local)
            print(f"{os.path.getsize(local)/1024:,.0f} KB")
        else:
            print(f"{name} already cached")
        return "file://" + os.path.abspath(local)
    return "file://" + os.path.abspath(path)


SUPERSTORE_SPARK = localise(SUPERSTORE_PATH, "SuperStoresKaggle.csv")
TELCO_SPARK      = localise(TELCO_PATH,      "TelcoCustomerKaggle.csv")

print("superstore ->", SUPERSTORE_SPARK)
print("telco      ->", TELCO_SPARK)




if HAS_SPARK:
    # Option A: inferSchema=True — convenient, but it READS THE FILE TWICE
    # (once to infer, once to load). On a 500 GB file that is unacceptable.
    sdf_infer = (spark.read
                 .option("header", True)
                 .option("inferSchema", True)
                 .csv(SUPERSTORE_SPARK))
    print("--- inferSchema=True ---")
    sdf_infer.printSchema()
else:
    print("skipped")

SuperStoresKaggle.csv already cached
TelcoCustomerKaggle.csv already cached
superstore -> file:///content/spark_data/SuperStoresKaggle.csv
telco      -> file:///content/spark_data/TelcoCustomerKaggle.csv
--- inferSchema=True ---
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)



In [137]:
if HAS_SPARK:
    # Option B: explicit schema — one pass, no guessing, and it FAILS LOUDLY
    # if the data changes shape. This is what production code does.
    schema = StructType([
        #            name                type      nullable
        StructField("Row ID",        IntegerType(), True),
        StructField("Order ID",      StringType(),  True),
        StructField("Order Date",    StringType(),  True),   # parse it ourselves
        StructField("Ship Date",     StringType(),  True),
        StructField("Ship Mode",     StringType(),  True),
        StructField("Customer ID",   StringType(),  True),
        StructField("Customer Name", StringType(),  True),
        StructField("Segment",       StringType(),  True),
        StructField("Country",       StringType(),  True),
        StructField("City",          StringType(),  True),
        StructField("State",         StringType(),  True),
        StructField("Postal Code",   StringType(),  True),   # identifier -> string
        StructField("Region",        StringType(),  True),
        StructField("Product ID",    StringType(),  True),
        StructField("Category",      StringType(),  True),
        StructField("Sub-Category",  StringType(),  True),
        StructField("Product Name",  StringType(),  True),
        StructField("Sales",         DoubleType(),  True),
    ])

    sdf = (spark.read.option("header", True).schema(schema).csv(SUPERSTORE_SPARK))
    print("--- explicit schema ---")
    sdf.printSchema()
    print("partitions:", sdf.rdd.getNumPartitions())
else:
    print("skipped")

--- explicit schema ---
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)

partitions: 1


In [138]:
if HAS_SPARK:
    # -----------------------------------------------------------------------
    # OPTION B: an explicit schema. This is what production code does.
    #
    # What it BUYS you:
    #   - ONE pass over the file instead of two
    #   - no type guessing, so the schema is stable across runs
    #   - the schema is documented, reviewable and version-controlled
    #
    # What it does NOT buy you:
    #   - loud failure on bad data. Spark's CSV reader defaults to
    #     mode="PERMISSIVE", which turns a type mismatch into a SILENT NULL.
    #     mode="FAILFAST" is what actually raises. See the modes table below.
    #
    # StructField takes three arguments:
    #     StructField(name, type, nullable)
    # nullable=True means nulls are allowed. Set it False and Spark treats the
    # column as guaranteed non-null — which enables optimisations, but gives
    # undefined behaviour if a null does arrive. True is the safe default.
    # -----------------------------------------------------------------------
    schema = StructType([
        #           name                  type        nullable
        StructField("Row ID",        IntegerType(),   True),
        StructField("Order ID",      StringType(),    True),
        StructField("Order Date",    StringType(),    True),   # see note 1
        StructField("Ship Date",     StringType(),    True),   # see note 1
        StructField("Ship Mode",     StringType(),    True),
        StructField("Customer ID",   StringType(),    True),
        StructField("Customer Name", StringType(),    True),
        StructField("Segment",       StringType(),    True),
        StructField("Country",       StringType(),    True),
        StructField("City",          StringType(),    True),
        StructField("State",         StringType(),    True),
        StructField("Postal Code",   StringType(),    True),   # see note 2
        StructField("Region",        StringType(),    True),
        StructField("Product ID",    StringType(),    True),
        StructField("Category",      StringType(),    True),
        StructField("Sub-Category",  StringType(),    True),
        StructField("Product Name",  StringType(),    True),
        StructField("Sales",         DoubleType(),    True),
    ])

    # NOTE 1 — dates land as StringType, NOT DateType.
    #   Spark's CSV reader wouldn't know the format is dd/MM/yyyy and would
    #   silently null the entire column. Land it as text, parse it explicitly
    #   later with F.to_date(col, "dd/MM/yyyy").
    #   This is the Day 5 rule: BRONZE HOLDS BYTES, SILVER ASSIGNS MEANING.
    #
    # NOTE 2 — Postal Code lands as StringType, NOT IntegerType.
    #   A postal code is an IDENTIFIER, not a quantity. Storing it as a number
    #   destroys leading zeros and permits nonsense like mean(Postal Code).
    #   Same fix as the pandas half, where it arrived as float64.


# PERMISSIVE (default)	null, silently	survives, quietly wrong
# DROPMALFORMED	—	vanishes, silently
# FAILFAST	—	raises an exception


    # sdf = (spark.read
    #        .option("header", True)          # skip row 1; names come from schema
    #       #  .option("mode", "FAILFAST")      # <<< THIS is what fails loudly
    #        .schema(schema)                  # <<< replaces inferSchema
    #        .csv(SUPERSTORE_SPARK))


    sdf = (spark.read
        .option("header", True)
        .option("quote", '"')            # field delimiter (this is the default)
        .option("escape", '"')           # <<< THE FIX: "" is an escaped quote
        .option("multiLine", True)       # allow newlines inside quoted fields
     #  .option("mode", "FAILFAST")      # <<< THIS is what fails loudly
        .schema(schema)
        .csv(SUPERSTORE_SPARK))

    print("--- explicit schema, FAILFAST ---")
    sdf.printSchema()
    print("rows      :", f"{sdf.count():,}")
    print("partitions:", sdf.rdd.getNumPartitions())
    print()
    print("ONE partition. That means NO parallelism — Spark's default split")
    print("size is 128 MB and this file is ~2 MB. See §2.7, where we inflate")
    print("the data so partitioning actually means something.")
else:
    print("skipped — run  !pip install -q pyspark  first")

--- explicit schema, FAILFAST ---
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)

rows      : 9,800
partitions: 1

ONE partition. That means NO parallelism — Spark's default split
size is 128 MB and this file is ~2 MB. See §2.7, where we inflate
the data so partitioning actually means something.


## 2.2 DIVERGENCE 1 — lazy evaluation

**In pandas, every line executes immediately.** In Spark, transformations build a **plan**. Nothing runs until you call an **action**.

In [139]:
if HAS_SPARK:
    import time

    t0 = time.perf_counter()
    plan = (sdf
            .filter(F.col("Sales") > 100)
            .withColumn("tax", F.col("Sales") * 0.18)
            .groupBy("Region")
            .agg(F.sum("Sales").alias("total")))
    t_transform = time.perf_counter() - t0

    t0 = time.perf_counter()
    result = plan.collect()                 # <<< THE ACTION. Now it runs.
    t_action = time.perf_counter() - t0

    print(f"building the plan (transformations): {t_transform:.5f}s")
    print(f"executing it (action)             : {t_action:.5f}s")
    print()
    print("The transformations returned instantly regardless of data size.")
    print("They did NO work — they described work.")
else:
    print("skipped")

building the plan (transformations): 0.14132s
executing it (action)             : 0.97130s

The transformations returned instantly regardless of data size.
They did NO work — they described work.


### Transformations vs actions

| Transformations (lazy) | Actions (trigger execution) |
|---|---|
| `select`, `filter`, `where` | `show()`, `collect()`, `take(n)` |
| `withColumn`, `drop` | `count()`, `first()` |
| `groupBy().agg()` | `write.parquet()` |
| `join`, `union` | `toPandas()` |
| `orderBy`, `distinct` | `foreach()` |

**Why this matters:** Spark sees the whole plan before running it, so it can reorder filters, prune columns, and skip files. That's the optimiser. `.explain()` shows you the result.

In [140]:
if HAS_SPARK:
    # The optimiser pushes the filter down and prunes unused columns
    (sdf.select("Region", "Sales", "Category")
        .filter(F.col("Sales") > 100)
        .groupBy("Region").agg(F.sum("Sales"))
        .explain(mode="simple"))
else:
    print("skipped")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Region#3434], functions=[sum(Sales#3439)])
   +- Exchange hashpartitioning(Region#3434, 8), ENSURE_REQUIREMENTS, [plan_id=3913]
      +- HashAggregate(keys=[Region#3434], functions=[partial_sum(Sales#3439)])
         +- Filter (isnotnull(Sales#3439) AND (Sales#3439 > 100.0))
            +- FileScan csv [Region#3434,Sales#3439] Batched: false, DataFilters: [isnotnull(Sales#3439), (Sales#3439 > 100.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/spark_data/SuperStoresKaggle.csv], PartitionFilters: [], PushedFilters: [IsNotNull(Sales), GreaterThan(Sales,100.0)], ReadSchema: struct<Region:string,Sales:double>




## 2.3 DIVERGENCE 2 — there is no index

pandas has an index: `.loc`, `set_index`, automatic row alignment on arithmetic between Series. **Spark has none of that.** Every operation is column-based, and row order is not guaranteed unless you `orderBy`.

In [141]:
if HAS_SPARK:
    print("pandas has an index:")
    print(" ", df.index[:5].tolist())
    print("  df.loc[0, 'Sales'] =", df.loc[0, "Sales"])
    print()
    print("Spark has no index, no .loc, no set_index, no row labels.")
    print("There is no 'first row' without an explicit orderBy.")
    print()
    print("Consequence: to reference a specific row you need a column that")
    print("identifies it. Here that's Row ID — a surrogate key, which is why")
    print("the Day 3 normalisation material matters in distributed systems.")
    sdf.filter(F.col("Row ID") == 1).select("Row ID", "Order ID", "Sales").show()
else:
    print("skipped")

pandas has an index:
  [0, 1, 2, 3, 4]
  df.loc[0, 'Sales'] = 261.96

Spark has no index, no .loc, no set_index, no row labels.
There is no 'first row' without an explicit orderBy.

Consequence: to reference a specific row you need a column that
identifies it. Here that's Row ID — a surrogate key, which is why
the Day 3 normalisation material matters in distributed systems.
+------+--------------+------+
|Row ID|      Order ID| Sales|
+------+--------------+------+
|     1|CA-2017-152156|261.96|
+------+--------------+------+



> **Column names with spaces work in `F.col("Row ID")`**, but they need backticks inside SQL expression strings (`F.expr("`Row ID` = 1")`) and they break `sdf.Row ID` attribute access entirely. Another reason production schemas use `snake_case`.

## 2.4 DIVERGENCE 3 — no `inplace`, everything is immutable

In [142]:
if HAS_SPARK:
    # pandas offers inplace= (and you should avoid it anyway)
    tmp = df.copy()
    # tmp  =tmp.drop(columns=["tax"])
    tmp.drop(columns=["tax"], inplace=True)
    print("pandas inplace=True works:", "tax" not in tmp.columns)
    print()
    # Spark DataFrames are IMMUTABLE. Every operation returns a NEW DataFrame.
    sdf2 = sdf.withColumn("tax", F.col("Sales") * 0.18)
    print("original columns :", len(sdf.columns))
    print("new DataFrame    :", len(sdf2.columns))
    print("original unchanged — you MUST reassign:  sdf = sdf.withColumn(...)")
else:
    print("skipped")

pandas inplace=True works: True

original columns : 18
new DataFrame    : 19
original unchanged — you MUST reassign:  sdf = sdf.withColumn(...)


## 2.5 DIVERGENCE 4 — `collect()` and `toPandas()` are dangerous

This is the one that takes down production jobs.

`collect()` pulls **every row** to the **driver** — a single JVM with a fixed heap. On 10 rows it's fine. On 500 million it's an `OutOfMemoryError` that kills the whole application.

`toPandas()` is the same thing wearing a friendlier name.

In [143]:
if HAS_SPARK:
    print("SAFE — bounded output:")
    sdf.select("Order ID", "Region", "Sales").show(5, truncate=False)

    print("SAFE — take(n) returns n rows to the driver:")
    for r in sdf.select("Order ID", "Sales").take(3):
        print("  ", r["Order ID"], r["Sales"])

    print()
    print("SAFE — aggregate first, THEN collect. The result is tiny.")
    for r in sdf.groupBy("Region").agg(F.sum("Sales").alias("total")).collect():
        print(f"   {r['Region']:<9} {r['total']:>12,.2f}")

    print()
    print("SAFE — limit before converting")
    small = sdf.select("Order ID", "Sales").limit(5).toPandas()
    print(f"   {len(small)} rows to pandas")

    print()
    print("DANGEROUS (do not run on real data):")
    print("   sdf.collect()      # every row -> driver heap")
    print("   sdf.toPandas()     # every row -> driver memory as a pandas frame")
    print("   len(sdf.collect()) # use sdf.count() instead")
else:
    print("skipped")

SAFE — bounded output:
+--------------+------+--------+
|Order ID      |Region|Sales   |
+--------------+------+--------+
|CA-2017-152156|South |261.96  |
|CA-2017-152156|South |731.94  |
|CA-2017-138688|West  |14.62   |
|US-2016-108966|South |957.5775|
|US-2016-108966|South |22.368  |
+--------------+------+--------+
only showing top 5 rows
SAFE — take(n) returns n rows to the driver:
   CA-2017-152156 261.96
   CA-2017-152156 731.94
   CA-2017-138688 14.62

SAFE — aggregate first, THEN collect. The result is tiny.
   East        669,518.73
   South       389,151.46
   Central     492,646.91
   West        710,219.68

SAFE — limit before converting
   5 rows to pandas

DANGEROUS (do not run on real data):
   sdf.collect()      # every row -> driver heap
   sdf.toPandas()     # every row -> driver memory as a pandas frame
   len(sdf.collect()) # use sdf.count() instead


## 2.6 The six operations, side by side

Same three answers as section 1, computed in Spark.

In [144]:
if HAS_SPARK:
    print("=" * 62)
    print("pandas :  df.info() / describe()")
    print("Spark  :  printSchema() / describe() / summary()")
    print("=" * 62)
    sdf.select("Sales").describe().show()
else:
    print("skipped")

pandas :  df.info() / describe()
Spark  :  printSchema() / describe() / summary()
+-------+------------------+
|summary|             Sales|
+-------+------------------+
|  count|              9800|
|   mean|230.76905945917886|
| stddev| 626.6518748388045|
|    min|             0.444|
|    max|          22638.48|
+-------+------------------+



In [145]:
if HAS_SPARK:
    # SELECT + WHERE
    print("pandas: df[(df.Sales > 1000) & (df.Region == 'West')]")
    print("Spark : .filter((col('Sales') > 1000) & (col('Region') == 'West'))")
    print()
    n_spark = sdf.filter((F.col("Sales") > 1000) & (F.col("Region") == "West")).count() # spark way
    n_pd = len(df[(df["Sales"] > 1000) & (df["Region"] == "West")])
    print(f"Spark count: {n_spark}   pandas count: {n_pd}   match: {n_spark == n_pd}") # pandas way
else:
    print("skipped")

pandas: df[(df.Sales > 1000) & (df.Region == 'West')]
Spark : .filter((col('Sales') > 1000) & (col('Region') == 'West'))

Spark count: 157   pandas count: 157   match: True


In [146]:
if HAS_SPARK:
    # GROUP BY + aggregate. Note countDistinct — the nunique() equivalent.
    print("Spark groupBy:")
    (sdf.groupBy("Region")
        .agg(F.round(F.sum("Sales"), 2).alias("total_sales"),
             F.countDistinct("Order ID").alias("orders"),
             F.count("*").alias("lines"),
             F.round(F.avg("Sales"), 2).alias("avg_line"))
        .orderBy(F.col("total_sales").desc())
        .show())

    print("pandas equivalent (from section 1.6):")
    print(region[["total_sales", "orders", "lines", "avg_line"]])
else:
    print("skipped")

Spark groupBy:
+-------+-----------+------+-----+--------+
| Region|total_sales|orders|lines|avg_line|
+-------+-----------+------+-----+--------+
|   West|  710219.68|  1587| 3140|  226.18|
|   East|  669518.73|  1369| 2785|   240.4|
|Central|  492646.91|  1156| 2277|  216.36|
|  South|  389151.46|   810| 1598|  243.52|
+-------+-----------+------+-----+--------+

pandas equivalent (from section 1.6):
         total_sales  orders  lines  avg_line
Region                                       
West       710219.68    1587   3140    226.18
East       669518.73    1369   2785    240.40
Central    492646.91    1156   2277    216.36
South      389151.46     810   1598    243.52


In [147]:
# if HAS_SPARK:
#     from pyspark.sql import functions as F

#     total = sdf.count()
#     nulls = sdf.filter(F.col("Sales").isNull()).count()

#     print(f"rows              : {total:,}")
#     print(f"Sales is NULL     : {nulls:,}  ({nulls/total:.2%})")
#     print()
#     print(f"Spark  SUM(Sales) : {sdf.agg(F.sum('Sales')).collect()[0][0]:,.2f}")
#     print(f"pandas SUM(Sales) : {df['Sales'].sum():,.2f}")
#     print()

#     # Read the SAME file with everything as text, and find the offenders
#     raw = (spark.read.option("header", True)
#            .csv(SUPERSTORE_SPARK))                    # no schema -> all strings

#     bad = raw.filter(F.col("Sales").cast("double").isNull() &
#                      F.col("Sales").isNotNull())

#     print(f"unparseable Sales values: {bad.count()}")
#     bad.select("Order ID", "Sales").show(20, truncate=False)

In [148]:
if HAS_SPARK:
    from pyspark.sql import functions as F

    # ANSI mode changes cast() behaviour: it RAISES on bad input instead of
    # returning null. Default in Spark 4.x and recent Databricks runtimes.

# ANSI mode	cast("double") on "abc"
# false (older Spark)	returns null
# true (Spark 4.x default)	raises NumberFormatException
    print("ANSI mode enabled:", spark.conf.get("spark.sql.ansi.enabled"))
    print()

    total = sdf.count()
    nulls = sdf.filter(F.col("Sales").isNull()).count()

    print(f"rows              : {total:,}")
    print(f"Sales is NULL     : {nulls:,}  ({nulls/total:.2%})")
    print()
    print(f"Spark  SUM(Sales) : {sdf.agg(F.sum('Sales')).collect()[0][0]:,.2f}")
    print(f"pandas SUM(Sales) : {df['Sales'].sum():,.2f}")
    print()

    # Read the SAME file as all-text, then find the offenders.
    # try_cast returns NULL on failure; cast() would RAISE under ANSI mode.
    raw = spark.read.option("header", True).csv(SUPERSTORE_SPARK)

    bad = raw.filter(F.expr("try_cast(Sales as double) IS NULL") &
                     F.col("Sales").isNotNull())

    print(f"unparseable Sales values: {bad.count()}")
    bad.select("Order ID", "Product Name", "Sales").show(20, truncate=False)
else:
    print("skipped")

ANSI mode enabled: true

rows              : 9,800
Sales is NULL     : 0  (0.00%)

Spark  SUM(Sales) : 2,261,536.78
pandas SUM(Sales) : 2,261,536.78

unparseable Sales values: 292
+--------------+---------------------------------------------------------------------------------+---------------+
|Order ID      |Product Name                                                                     |Sales          |
+--------------+---------------------------------------------------------------------------------+---------------+
|US-2016-150630|"Acco Pressboard Covers with Storage Hooks, 14 7/8"" x 11""                      | Executive Red"|
|US-2018-118038|"6"" Cubicle Wall Clock                                                          | Black"        |
|US-2016-156867|"Avery Trapezoid Ring Binder, 3"" Capacity                                       | Black         |
|US-2018-124303|"Pressboard Covers with Storage Hooks, 9 1/2"" x 11""                            | Light Blue"   |
|US-2016-159982

In [149]:
if HAS_SPARK:
    # withColumn — derived columns. to_date + datediff for the ship delay.
    sdf_enriched = (sdf
        .withColumn("order_dt", F.to_date("Order Date", "dd/MM/yyyy"))
        .withColumn("ship_dt",  F.to_date("Ship Date",  "dd/MM/yyyy"))
        .withColumn("ship_delay", F.datediff("ship_dt", "order_dt"))
        .withColumn("late", F.when(F.col("ship_delay") > 5, 1).otherwise(0))
        .withColumn("year", F.year("order_dt")))

    sdf_enriched.select("Order ID", "order_dt", "ship_dt", "ship_delay", "late").show(5)

    late_rate = sdf_enriched.agg(F.avg("late")).collect()[0][0]
    print(f"Spark late rate : {late_rate:.1%}")
    print(f"pandas late rate: {df['late'].mean():.1%}")
else:
    print("skipped")

+--------------+----------+----------+----------+----+
|      Order ID|  order_dt|   ship_dt|ship_delay|late|
+--------------+----------+----------+----------+----+
|CA-2017-152156|2017-11-08|2017-11-11|         3|   0|
|CA-2017-152156|2017-11-08|2017-11-11|         3|   0|
|CA-2017-138688|2017-06-12|2017-06-16|         4|   0|
|US-2016-108966|2016-10-11|2016-10-18|         7|   1|
|US-2016-108966|2016-10-11|2016-10-18|         7|   1|
+--------------+----------+----------+----------+----+
only showing top 5 rows
Spark late rate : 18.2%
pandas late rate: 18.2%


> **`F.when(...).otherwise(...)`** is Spark's `CASE WHEN`. It's also pandas' `np.where`. Same idea, three syntaxes — this is why the SQL grounding pays off.

In [150]:
if HAS_SPARK:
    # JOIN — same how= semantics as pandas and SQL
    s_orders = sdf.select("Order ID", "Order Date", "Ship Mode", "Customer ID").dropDuplicates(["Order ID"])
    s_lines  = sdf.select("Row ID", "Order ID", "Product ID", "Sales")

    print("orders:", s_orders.count(), " lines:", s_lines.count())

    joined = s_lines.join(s_orders, on="Order ID", how="left")
    print("left join:", joined.count(), "<<< line grain preserved")
    print()
    print("Join types: inner, left, right, outer, left_semi, left_anti, cross")
    print()
    print("left_anti IS the anti-join — no need for the isna() trick:")
    fake = s_lines.limit(3).withColumn("Order ID", F.lit("XX-9999-999999"))
    print("orphan rows found:", fake.join(s_orders, on="Order ID", how="left_anti").count())
else:
    print("skipped")

orders: 4922  lines: 9800
left join: 9800 <<< line grain preserved

Join types: inner, left, right, outer, left_semi, left_anti, cross

left_anti IS the anti-join — no need for the isna() trick:
orphan rows found: 3


> **Spark has `left_semi` and `left_anti` as first-class join types** — the SEMI JOIN and ANTI JOIN patterns from Day 1, built into the API instead of emulated with `IN` / `NOT EXISTS`. Worth knowing: interviewers like this one.

## 2.7 Making the scale realistic

9,800 rows is too small for Spark to look like anything but ceremony. Inflate it so partitions, shuffles and plans mean something.

In [151]:
if HAS_SPARK:
    import time

    # 128-deep union plan (which can choke the query planner).

    # What I originally wrote — DON'T
    # big = reduce(lambda a, b: a.union(b), [sdf] * 128)

    # crossJoin against spark.range replicates the data without building a
    big = (sdf.crossJoin(spark.range(128).withColumnRenamed("id", "_copy"))
              .drop("_copy")
              .repartition(16))


    # repartition(n) does a full shuffle and can increase or decrease partition count.
    # coalesce(n) avoids the shuffle but can only reduce. That distinction is a Day 6 topic and a standard interview question.

    n = big.count()
    print("rows      :", f"{n:,}")
    print("partitions:", big.rdd.getNumPartitions())
    print()

    t0 = time.perf_counter()
    agg = big.groupBy("Region", "Category").agg(F.sum("Sales").alias("total"))
    print(f"transformation: {time.perf_counter()-t0:.5f}s  (still instant — lazy)")

    t0 = time.perf_counter()
    rows = agg.collect()
    print(rows)
    print(f"action        : {time.perf_counter()-t0:.3f}s  "
          f"({len(rows)} result rows from {n:,} scanned)")
else:
    print("skipped")

rows      : 1,254,400
partitions: 16

transformation: 0.27267s  (still instant — lazy)
[Row(Region='Central', Category='Office Supplies', total=20939551.104000084), Row(Region='East', Category='Technology', total=33678915.45599945), Row(Region='Central', Category='Technology', total=21598618.623999752), Row(Region='West', Category='Technology', total=31667831.039999783), Row(Region='West', Category='Office Supplies', total=27835713.15199918), Row(Region='West', Category='Furniture', total=31404575.423999544), Row(Region='South', Category='Furniture', total=14916029.439999936), Row(Region='Central', Category='Furniture', total=20520635.161599807), Row(Region='East', Category='Office Supplies', total=25592423.807999607), Row(Region='South', Category='Office Supplies', total=15926370.687999906), Row(Region='South', Category='Technology', total=18968986.623999596), Row(Region='East', Category='Furniture', total=26427057.66399956)]
action        : 11.072s  (12 result rows from 1,254,400 sca

In [152]:
if HAS_SPARK:
    # cache() when you reuse a DataFrame. Without it, every action re-reads
    # and re-computes the ENTIRE lineage from source.
    import time

    big.unpersist()
    t0 = time.perf_counter(); big.count(); t_cold1 = time.perf_counter() - t0
    t0 = time.perf_counter(); big.count(); t_cold2 = time.perf_counter() - t0

    big.cache()
    big.count()                                    # materialise the cache
    t0 = time.perf_counter(); big.count(); t_warm = time.perf_counter() - t0

    print(f"uncached, 1st : {t_cold1:.3f}s")
    print(f"uncached, 2nd : {t_cold2:.3f}s   <<< recomputed from scratch")
    print(f"cached        : {t_warm:.3f}s")
    print()
    print("Rule: cache a DataFrame you touch more than once. Unpersist when done.")
    big.unpersist()
else:
    print("skipped")

uncached, 1st : 2.695s
uncached, 2nd : 3.735s   <<< recomputed from scratch
cached        : 0.574s

Rule: cache a DataFrame you touch more than once. Unpersist when done.


In [153]:
if HAS_SPARK:
    import time
    from pyspark.sql import functions as F

    # ---- Build a DataFrame with an EXPENSIVE lineage ----------------------
    # Several shuffles and a self-join, so recomputation genuinely costs.
    expensive = (big
        .groupBy("Region", "Category", "Sub-Category")
        .agg(F.sum("Sales").alias("total"),
             F.countDistinct("Order ID").alias("orders"),
             F.avg("Sales").alias("avg_sale"))
        .withColumn("rank", F.rank().over(
            __import__("pyspark").sql.Window
              .partitionBy("Region").orderBy(F.col("total").desc())))
        .join(big.groupBy("Region").agg(F.sum("Sales").alias("region_total")),
              on="Region", how="left")
        .withColumn("share", F.col("total") / F.col("region_total")))

    def timed(label, fn):
        t0 = time.perf_counter()
        result = fn()
        return time.perf_counter() - t0, result

    # ---- UNCACHED: three actions, each rebuilding the whole lineage ------
    expensive.unpersist()
    t1, _ = timed("a", lambda: expensive.agg(F.sum("total")).collect())
    t2, _ = timed("b", lambda: expensive.filter(F.col("rank") <= 2).count())
    t3, _ = timed("c", lambda: expensive.agg(F.avg("share")).collect())
    uncached = t1 + t2 + t3

    # ---- CACHED -----------------------------------------------------------
    expensive.cache()
    t_mat, _ = timed("materialise", lambda: expensive.count())   # populate it

    c1, _ = timed("a", lambda: expensive.agg(F.sum("total")).collect())
    c2, _ = timed("b", lambda: expensive.filter(F.col("rank") <= 2).count())
    c3, _ = timed("c", lambda: expensive.agg(F.avg("share")).collect())
    cached = c1 + c2 + c3

    print("THREE ACTIONS ON THE SAME DATAFRAME")
    print("=" * 58)
    print(f"UNCACHED   action a {t1:.3f}s   b {t2:.3f}s   c {t3:.3f}s")
    print(f"           total    {uncached:.3f}s   <<< lineage rebuilt 3 times")
    print()
    print(f"CACHED     materialise {t_mat:.3f}s  (pays the cost ONCE)")
    print(f"           action a {c1:.3f}s   b {c2:.3f}s   c {c3:.3f}s")
    print(f"           total    {cached:.3f}s")
    print()
    print(f"speedup on the reuse: {uncached/max(cached, 1e-9):.1f}x")
    print()
    print("The saving grows with (cost of lineage) x (number of reuses).")
    print("On a trivial lineage used once, cache() is pure overhead.")

    expensive.unpersist()
else:
    print("skipped")

THREE ACTIONS ON THE SAME DATAFRAME
UNCACHED   action a 7.839s   b 4.850s   c 7.528s
           total    20.217s   <<< lineage rebuilt 3 times

CACHED     materialise 11.263s  (pays the cost ONCE)
           action a 0.743s   b 0.904s   c 0.770s
           total    2.417s

speedup on the reuse: 8.4x

The saving grows with (cost of lineage) x (number of reuses).
On a trivial lineage used once, cache() is pure overhead.


## 2.8 Section 2 recap — the translation table

| Operation | pandas | PySpark |
|---|---|---|
| Inspect schema | `.info()` | `.printSchema()` |
| Summary stats | `.describe()` | `.describe()` / `.summary()` |
| Select columns | `df[["a","b"]]` | `.select("a","b")` |
| Filter rows | `df[df.a > 5]` | `.filter(F.col("a") > 5)` |
| New column | `df["c"] = ...` | `.withColumn("c", ...)` |
| Conditional | `np.where` | `F.when().otherwise()` |
| Group + aggregate | `.groupby().agg()` | `.groupBy().agg()` |
| Count distinct | `.nunique()` | `F.countDistinct()` |
| Join | `.merge(how=)` | `.join(on=, how=)` |
| Anti-join | `left` + `.isna()` | `how="left_anti"` |
| Drop duplicates | `.drop_duplicates()` | `.dropDuplicates()` |
| Fill nulls | `.fillna()` | `.fillna()` / `.na.fill()` |
| Sort | `.sort_values()` | `.orderBy()` |
| Preview | `.head()` | `.show()` |
| Row count | `len(df)` | `.count()` |
| Parse date | `pd.to_datetime` | `F.to_date(col, fmt)` |
| Stack rows | `pd.concat` | `.union()` |

**The four divergences, in interview-answer form**

1. **Lazy** — transformations build a plan; only actions execute. Enables whole-query optimisation.
2. **No index** — no `.loc`, no row alignment. Everything is column-based, order needs `orderBy`.
3. **Immutable** — no `inplace`; every operation returns a new DataFrame, so reassign.
4. **`collect()` / `toPandas()` pull everything to the driver** — aggregate or `limit` first.

**Plus two that come up constantly:** `cache()` anything you reuse, and never trust `inferSchema` in production.

---

# 3. pandas → scikit-learn (DS live · DE observes · 25 min)

**DE students:** you're observing, but the vocabulary here — leakage, train/test split, precision vs recall — appears in DE interviews whenever a pipeline feeds a model. Know what these words mean.

**DS students:** the centrepiece is **why `Pipeline` exists**. Not the syntax — the reason.

New dataset: **Telco Customer Churn.** 7,043 rows, one per customer, and a real target.

In [154]:
telco = pd.read_csv(TELCO_PATH)
print("shape:", telco.shape)
telco.head(3)

shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [155]:
telco.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## 3.1 Diagnosis: find the defect

### Predict first

`MonthlyCharges` is `float64`. Look at `TotalCharges` — same kind of column, same kind of values. What dtype is it, and why?

In [156]:
print("MonthlyCharges:", telco["MonthlyCharges"].dtype)
print("TotalCharges  :", telco["TotalCharges"].dtype, "   <<< TEXT, not numeric")
print()
print("nulls reported by pandas:", telco["TotalCharges"].isna().sum())
print("... so what's wrong with it?")
print()
print("sample values:", telco["TotalCharges"].head(4).tolist())

MonthlyCharges: float64
TotalCharges  : object    <<< TEXT, not numeric

nulls reported by pandas: 0
... so what's wrong with it?

sample values: ['29.85', '1889.5', '108.15', '1840.75']


In [157]:
# Coerce to numeric and see what fails
tc = pd.to_numeric(telco["TotalCharges"], errors="coerce")
print("unparseable values:", tc.isna().sum())
print()
# repr() reveals what the raw values actually are
bad_values = telco.loc[tc.isna(), "TotalCharges"].unique()
print("the offending values:", [repr(v) for v in bad_values])

unparseable values: 11

the offending values: ["' '"]


> **Eleven values are a single space `' '`.** Not `NaN`, not empty — a space. `read_csv` sees a non-numeric character in the column and types the whole thing as text. **One bad value in 7,043 changes the dtype of the entire column.**
>
> This is the single most common real-world dtype bug, and `.isna()` alone would never have found it.

### Now the important part: *why* are those 11 blank?

In [158]:
# Never fill a null before you understand it. Look at the rows.
blanks = telco[tc.isna()]
print("rows with blank TotalCharges:", len(blanks))
print()
print("their tenure values:", blanks["tenure"].unique().tolist())
print("their MonthlyCharges:", blanks["MonthlyCharges"].head(3).tolist())
print("their Churn:", blanks["Churn"].unique().tolist())
print()
blanks[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]].head(11)

rows with blank TotalCharges: 11

their tenure values: [0]
their MonthlyCharges: [52.55, 20.25, 80.85]
their Churn: ['No']



,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


> ### The answer changes the fix
>
> **All 11 have `tenure = 0`.** They are brand-new customers who have not been billed yet. Their total charges aren't *unknown* — they are genuinely **zero**.
>
> | Fill strategy | Result |
> |---|---|
> | `fillna(median)` | ~1,400 — **invents revenue that doesn't exist** |
> | `fillna(mean)` | ~2,280 — worse |
> | `dropna()` | loses 11 real customers |
> | **`fillna(0)`** | **correct — reflects reality** |
>
> Every one of those options is a single line of code. Only one is right, and the code can't tell you which. **You have to look at the rows.**

In [159]:
telco["TotalCharges"] = tc.fillna(0.0)
print("dtype:", telco["TotalCharges"].dtype)
print("nulls:", telco["TotalCharges"].isna().sum())
print("min  :", telco["TotalCharges"].min())
print()
print("total nulls in the whole frame:", telco.isna().sum().sum())
print("duplicate rows               :", telco.duplicated().sum())
print("distinct customerID          :", telco["customerID"].nunique(), f"of {len(telco)}")

dtype: float64
nulls: 0
min  : 0.0

total nulls in the whole frame: 0
duplicate rows               : 0
distinct customerID          : 7043 of 7043


## 3.2 The target, and the accuracy trap

In [160]:
print(telco["Churn"].value_counts())
print()
print(telco["Churn"].value_counts(normalize=True).round(4))
print()
baseline = (telco["Churn"] == "No").mean()
print(f"A model that ALWAYS predicts 'No' scores {baseline:.1%} accuracy")
print("and catches ZERO churners. It is worthless and looks good.")
print()
print("This is why accuracy is the wrong metric on imbalanced data.")
print("Later lessons opens with the same problem at 95%.")

Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn
No     0.7346
Yes    0.2654
Name: proportion, dtype: float64

A model that ALWAYS predicts 'No' scores 73.5% accuracy
and catches ZERO churners. It is worthless and looks good.

This is why accuracy is the wrong metric on imbalanced data.
Later lessons opens with the same problem at 95%.


In [161]:
# Quick signal check before modelling — does the data contain anything learnable?
for col in ["Contract", "InternetService", "PaymentMethod", "TechSupport"]:
    rates = telco.groupby(col, observed=True)["Churn"].apply(lambda s: (s == "Yes").mean())
    print(f"{col}:")
    for k, v in rates.sort_values(ascending=False).items():
        print(f"   {str(k):<28} {v:.1%}")
    print()


# Does the payment effect survive controlling for contract?
# pd.crosstab(telco["PaymentMethod"], telco["Contract"],
#             values=telco["Churn"], aggfunc="mean").round(3)

Contract:
   Month-to-month               42.7%
   One year                     11.3%
   Two year                     2.8%

InternetService:
   Fiber optic                  41.9%
   DSL                          19.0%
   No                           7.4%

PaymentMethod:
   Electronic check             45.3%
   Mailed check                 19.1%
   Bank transfer (automatic)    16.7%
   Credit card (automatic)      15.2%

TechSupport:
   No                           41.6%
   Yes                          15.2%
   No internet service          7.4%



> Month-to-month contracts churn at **42.7%**, two-year at **2.8%**. That is enormous signal — a model should find it easily, and `Contract` will dominate feature importance. Worth knowing *before* you train, so an implausible result is recognisable.

## 3.3 Feature types — and two judgement calls

In [162]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = [c for c in telco.columns
                    if c not in numeric_cols + ["customerID", "Churn", "SeniorCitizen"]]

print("numeric    :", numeric_cols)
print()
print(f"categorical ({len(categorical_cols)}):")
for c in categorical_cols:
    print(f"   {c:<20} {telco[c].nunique()} values: {sorted(telco[c].unique())[:4]}")

numeric    : ['tenure', 'MonthlyCharges', 'TotalCharges']

categorical (15):
   gender               2 values: ['Female', 'Male']
   Partner              2 values: ['No', 'Yes']
   Dependents           2 values: ['No', 'Yes']
   PhoneService         2 values: ['No', 'Yes']
   MultipleLines        3 values: ['No', 'No phone service', 'Yes']
   InternetService      3 values: ['DSL', 'Fiber optic', 'No']
   OnlineSecurity       3 values: ['No', 'No internet service', 'Yes']
   OnlineBackup         3 values: ['No', 'No internet service', 'Yes']
   DeviceProtection     3 values: ['No', 'No internet service', 'Yes']
   TechSupport          3 values: ['No', 'No internet service', 'Yes']
   StreamingTV          3 values: ['No', 'No internet service', 'Yes']
   StreamingMovies      3 values: ['No', 'No internet service', 'Yes']
   Contract             3 values: ['Month-to-month', 'One year', 'Two year']
   PaperlessBilling     2 values: ['No', 'Yes']
   PaymentMethod        4 values: ['Bank tra

### Judgement call 1 — `SeniorCitizen`

It's stored as `int64` with values 0 and 1. **Is it numeric or categorical?**

It's already a valid binary indicator, so treating it as numeric works and needs no encoding. But scaling it (mean 0, sd 1) makes it meaningless to interpret. Cleanest choice: leave it as-is and pass it through untouched.

### Judgement call 2 — `'No internet service'`

Six columns use it as a third category. It's **redundant** — it's fully determined by `InternetService == 'No'`. You can collapse it to `'No'` (fewer dummy columns, loses nothing) or keep it (one-hot handles it fine).

Neither is wrong. **Being able to explain the tradeoff is what the interview tests.**

In [163]:
# Collapse the redundant category and prove nothing is lost
service_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]

before = sum(telco[c].nunique() for c in service_cols)
telco_clean = telco.copy()
for c in service_cols:
    telco_clean[c] = telco_clean[c].replace({"No internet service": "No"})
after = sum(telco_clean[c].nunique() for c in service_cols)

print(f"distinct values across the 6 service columns: {before} -> {after}")
print("dummy columns saved after one-hot:", before - after)
print()
# The information is preserved — it lives in InternetService
print(telco_clean.groupby("InternetService", observed=True)["TechSupport"]
      .value_counts().head(6))

distinct values across the 6 service columns: 18 -> 12
dummy columns saved after one-hot: 6

InternetService  TechSupport
DSL              No             1243
                 Yes            1178
Fiber optic      No             2230
                 Yes             866
No               No             1526
Name: count, dtype: int64


## 3.4 The split — before anything else touches the data

In [164]:
from sklearn.model_selection import train_test_split

X = telco_clean.drop(columns=["customerID", "Churn"])
y = (telco_clean["Churn"] == "Yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,        # reproducibility — always set it
    stratify=y,             # preserve the 26.5% churn rate in BOTH halves
)

print(f"train: {X_train.shape}   churn rate {y_train.mean():.4f}")
print(f"test : {X_test.shape}   churn rate {y_test.mean():.4f}")
print()
print("Without stratify=y, a random split could hand you a test set with")
print("22% or 31% churn, and your metrics would move for no real reason.")

train: (5282, 19)   churn rate 0.2654
test : (1761, 19)   churn rate 0.2652

Without stratify=y, a random split could hand you a test set with
22% or 31% churn, and your metrics would move for no real reason.


## 3.5 fit / transform / predict

The contract behind every sklearn object:

| Method | Does |
|---|---|
| `fit(X)` | **learns parameters** from the data and stores them |
| `transform(X)` | **applies** the stored parameters |
| `fit_transform(X)` | both — **only ever on training data** |
| `predict(X)` | produces outputs from a fitted model |

**The rule: `fit` sees training data only.** Everything below is a consequence of that one sentence.

In [165]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train[numeric_cols])          # LEARNS mean and sd — from train only

print("learned means:", dict(zip(numeric_cols, scaler.mean_.round(2))))
print("learned sds  :", dict(zip(numeric_cols, scaler.scale_.round(2))))
print()

# transform APPLIES those stored numbers — it does not re-learn
tr = scaler.transform(X_train[numeric_cols])
te = scaler.transform(X_test[numeric_cols])       # <<< transform, NOT fit_transform

print(f"train after scaling: mean {tr.mean():.6f}  sd {tr.std():.4f}   (0 and 1 by construction)")
print(f"test  after scaling: mean {te.mean():.6f}  sd {te.std():.4f}   (close, not exact — correct)")
print()
print("The test set is NOT exactly mean 0. If it were, the scaler had seen it.")

learned means: {'tenure': np.float64(32.36), 'MonthlyCharges': np.float64(64.86), 'TotalCharges': np.float64(2292.08)}
learned sds  : {'tenure': np.float64(24.63), 'MonthlyCharges': np.float64(30.17), 'TotalCharges': np.float64(2284.95)}

train after scaling: mean -0.000000  sd 1.0000   (0 and 1 by construction)
test  after scaling: mean -0.010955  sd 0.9819   (close, not exact — correct)

The test set is NOT exactly mean 0. If it were, the scaler had seen it.


## 3.6 Data leakage — the centrepiece

**Leakage** is when information from outside the training set influences the model. The result is a score that looks great and collapses in production.

Watch it happen.

In [166]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore")

# Prepare one-hot features once, so the ONLY difference between the two
# versions below is WHEN the scaler is fitted.
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first")
cat_all = enc.fit_transform(X[categorical_cols])

num_all = X[numeric_cols + ["SeniorCitizen"]].to_numpy(dtype=float)

idx_train, idx_test = train_test_split(
    np.arange(len(X)), test_size=0.25, random_state=42, stratify=y)

print("features:", cat_all.shape[1], "categorical dummies +", num_all.shape[1], "numeric")

features: 20 categorical dummies + 4 numeric


In [167]:
# ---------- THE WRONG WAY: scale everything, THEN split ----------
scaler_bad = StandardScaler()
num_scaled_bad = scaler_bad.fit_transform(num_all)      # <<< SAW THE TEST SET

Xb = np.hstack([num_scaled_bad, cat_all])
mb = LogisticRegression(max_iter=2000).fit(Xb[idx_train], y.iloc[idx_train])
auc_leaky = roc_auc_score(y.iloc[idx_test], mb.predict_proba(Xb[idx_test])[:, 1])

# roc_auc_score(y.iloc[idx_test], ...) — compares those predicted probabilities
#  against the true test labels (y.iloc[idx_test]) and returns the ROC-AUC: how
#  well the model ranks actual churners above actual non-churners, where 1.0 is perfect and 0.5 is random.


# : on the left means "all rows" (a colon by itself means "everything along this axis")
# 1 on the right means "column at index 1" — and remember indexing starts at 0, so index 1 is the second column

# ---------- THE RIGHT WAY: split, then fit on train only ----------
scaler_good = StandardScaler()
num_tr = scaler_good.fit_transform(num_all[idx_train])  # fit on TRAIN
num_te = scaler_good.transform(num_all[idx_test])       # transform TEST

Xg_tr = np.hstack([num_tr, cat_all[idx_train]])
Xg_te = np.hstack([num_te, cat_all[idx_test]])
mg = LogisticRegression(max_iter=2000).fit(Xg_tr, y.iloc[idx_train])
auc_clean = roc_auc_score(y.iloc[idx_test], mg.predict_proba(Xg_te)[:, 1])

# Receiver Operating Characteristic ROC
# Area Under the Curve. AUC
print(f"leaky  (scaled before split): ROC-AUC {auc_leaky:.5f}")
print(f"correct (scaled after split): ROC-AUC {auc_clean:.5f}")
print(f"difference                  : {auc_leaky - auc_clean:+.5f}")



# +0.00007 | The sign is positive, and that's the expected direction.
# Leakage inflates the score — the model looked slightly better than it honestly should have,

leaky  (scaled before split): ROC-AUC 0.84670
correct (scaled after split): ROC-AUC 0.84663
difference                  : +0.00007


> ### Read the numbers carefully — this is the real lesson
>
> The gap is **tiny**. `StandardScaler` leaks only a mean and a standard deviation, and with 7,043 rows those barely differ between the full data and the training 75%.
>
> **That is exactly why leakage is dangerous.** It doesn't announce itself with an implausible 0.99 AUC. It produces a small, invisible optimism that you cannot detect by looking at the score.
>
> And it gets much worse with:
> - **`SimpleImputer`** — the fill value is computed from test rows too
> - **Target encoding** — the target itself leaks into features
> - **`SMOTE` / resampling** — synthetic rows built from test data
> - **Feature selection** on the full dataset
> - **Time series** split randomly instead of chronologically
>
> With any of those, "small and invisible" becomes "large and catastrophic".
>
> **So the fix cannot be discipline.** It has to be structural.

## 3.7 `Pipeline` — making leakage structurally impossible

A `Pipeline` chains preprocessing and model into **one estimator**. When you call `pipeline.fit(X_train, y_train)`, *every* step fits on training data only. When you call `predict`, every step transforms.

You cannot leak by accident, because there is no separate step to get wrong.

In [168]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# ColumnTransformer: different preprocessing per column group.
# 3 numeric columns need imputing + scaling.
# 15 categorical columns need one-hot encoding.
# 1 column (SeniorCitizen) is already a clean binary and passes through.
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale",  StandardScaler()),
        ]), numeric_cols),

        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"),
         categorical_cols),
    ],
    remainder="passthrough",      # SeniorCitizen goes through untouched
    verbose_feature_names_out=False,
)

model = Pipeline([
    ("prep", preprocessor),
    ("clf",  LogisticRegression(max_iter=2000, class_weight="balanced")),
])

model

Pipeline(steps=[('prep',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['tenure', 'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['gender', 'Partner',
                                                   'Dependents', 'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod'])],
                                   verbose_feature_names_out=False)),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=2000))])

In [169]:
# ONE call fits everything, in the right order, on training data only
# 1. SimpleImputer     → learns the median of each numeric column
# 2. StandardScaler    → learns the mean and sd of each numeric column
# 3. OneHotEncoder     → learns which categories exist in each text column
# 4. (passthrough)     → nothing to learn
# 5. LogisticRegression → learns 24 coefficients + 1 intercept

model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("features after preprocessing:",
      model.named_steps["prep"].transform(X_train.head(1)).shape[1])
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))

# What ROC-AUC means, in plain terms: pick a random churner and a random stayer.
#  The probability that the model gave the churner a higher score is 84.6%.


# custom threshold of 0.3 instead of 0.5
y_pred_custom = (y_proba >= 0.3).astype(int)


features after preprocessing: 24
ROC-AUC: 0.8463


In [170]:
new_customer = pd.DataFrame([{
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 3,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 95.7,
    "TotalCharges": 287.1,
}])

proba = model.predict_proba(new_customer)[:, 1][0]
label = model.predict(new_customer)[0]

print(f"churn probability: {proba:.3f}")
print(f"prediction: {'WILL churn' if label == 1 else 'will stay'}")



churn probability: 0.895
prediction: WILL churn


> **`handle_unknown="ignore"` is not optional.** If a category appears in production that wasn't in training, the default behaviour is to *raise*. With `ignore`, the unseen category becomes all-zeros. A crashed prediction service is worse than a slightly degraded one.
>
> **`drop="first"`** removes one dummy per feature to avoid perfect collinearity — it matters for linear models, not for trees.

## 3.8 Evaluation — enough to set up Day 7

In [171]:
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, average_precision_score,
                             accuracy_score)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion matrix")
print("                  predicted")
print("                  No     Yes")
print(f"actual  No    {tn:>6} {fp:>7}")
print(f"        Yes   {fn:>6} {tp:>7}")
print()
print(f"accuracy          {accuracy_score(y_test, y_pred):.4f}")
print(f"baseline (all No) {(y_test == 0).mean():.4f}   <<< the bar to beat")
print()
print(classification_report(y_test, y_pred,
                            target_names=["stayed", "churned"], digits=3))
print(f"ROC-AUC : {roc_auc_score(y_test, y_proba):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test, y_proba):.4f}   <<< better for imbalance")

Confusion matrix
                  predicted
                  No     Yes
actual  No       949     345
        Yes       95     372

accuracy          0.7501
baseline (all No) 0.7348   <<< the bar to beat

              precision    recall  f1-score   support

      stayed      0.909     0.733     0.812      1294
     churned      0.519     0.797     0.628       467

    accuracy                          0.750      1761
   macro avg      0.714     0.765     0.720      1761
weighted avg      0.806     0.750     0.763      1761

ROC-AUC : 0.8463
PR-AUC  : 0.6372   <<< better for imbalance


### The business question behind the metrics

| | Meaning | Cost of getting it wrong |
|---|---|---|
| **Precision** | of those we flagged, how many really churn | wasted retention spend on happy customers |
| **Recall** | of those who churn, how many did we catch | lost customers you could have saved |

`class_weight="balanced"` pushed the model toward **recall** — it flags more customers, catching more real churners at the cost of more false alarms.

**Whether that's right is a business decision, not a modelling one.** If a retention offer costs ₹200 and a lost customer costs ₹8,000, high recall is obviously correct. Reverse those numbers and it isn't.

Day 7 goes deep on this. Today just know: **the threshold is a business lever, not fixed at 0.5.**

In [172]:
# Moving the threshold trades precision against recall. Same model throughout.
from sklearn.metrics import precision_score, recall_score, f1_score

print(f"{'threshold':>10} {'precision':>10} {'recall':>8} {'f1':>7} {'flagged':>8}")
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (y_proba >= t).astype(int)
    print(f"{t:>10.1f} {precision_score(y_test, pred, zero_division=0):>10.3f}"
          f" {recall_score(y_test, pred):>8.3f} {f1_score(y_test, pred):>7.3f}"
          f" {pred.sum():>8}")
print()
print("One trained model, five different business behaviours.")

 threshold  precision   recall      f1  flagged
       0.3      0.435    0.929   0.593      997
       0.4      0.472    0.865   0.611      856
       0.5      0.519    0.797   0.628      717
       0.6      0.553    0.717   0.624      606
       0.7      0.610    0.604   0.607      462

One trained model, five different business behaviours.


In [173]:
# Cross-validation on the PIPELINE — the scaler refits inside every fold.
# Doing CV on pre-scaled data leaks across folds, which is subtler than
# the split-level leak in 3.6 and just as wrong.
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")

print("fold scores:", scores.round(4))
print(f"mean {scores.mean():.4f}  sd {scores.std():.4f}")
print()
print("A tight sd means the estimate is stable. A wide one means the")
print("single train/test number above was partly luck.")

fold scores: [0.855  0.846  0.8628 0.8245 0.8364]
mean 0.8449  sd 0.0135

A tight sd means the estimate is stable. A wide one means the
single train/test number above was partly luck.


In [174]:
# What the model learned — sanity-check it against the signal from 3.2
import numpy as np

feat_names = model.named_steps["prep"].get_feature_names_out()
coefs = model.named_steps["clf"].coef_[0]

top = pd.DataFrame({"feature": feat_names, "coef": coefs})
top["abs"] = top["coef"].abs()
top = top.sort_values("abs", ascending=False).head(12)

print("Strongest drivers (positive = more likely to churn):")
for _, r in top.iterrows():
    bar = "+" if r["coef"] > 0 else "-"
    print(f"  {bar} {r['feature']:<38} {r['coef']:>7.3f}")
print()
print("Sanity check: does this match the churn rates from 3.2?")
print("Contract and tenure should dominate. If they don't, something is wrong.")

Strongest drivers (positive = more likely to churn):
  - Contract_Two year                       -1.359
  - tenure                                  -1.161
  - InternetService_No                      -1.101
  + InternetService_Fiber optic              0.986
  - Contract_One year                       -0.722
  + TotalCharges                             0.515
  - OnlineSecurity_Yes                      -0.389
  + PaperlessBilling_Yes                     0.373
  + PaymentMethod_Electronic check           0.367
  + StreamingMovies_Yes                      0.365
  - PhoneService_Yes                        -0.335
  - TechSupport_Yes                         -0.327

Sanity check: does this match the churn rates from 3.2?
Contract and tenure should dominate. If they don't, something is wrong.


## 3.9 Section 3 recap

| pandas | scikit-learn |
|---|---|
| `fillna(median)` | `SimpleImputer(strategy="median")` |
| manual z-score | `StandardScaler()` |
| `pd.get_dummies()` | `OneHotEncoder()` |
| different treatment per column | `ColumnTransformer` |
| the whole sequence | `Pipeline` |

> **Why `OneHotEncoder` and not `pd.get_dummies()`:** `get_dummies` has no `fit`. It has no memory of which columns existed at training time, so a new category in production silently changes your feature count and the model breaks. `OneHotEncoder` remembers.

**The five things to remember**

1. **One bad value changes a column's dtype.** `.info()` finds it; `.isna()` does not.
2. **Understand a null before filling it.** All 11 blanks had `tenure = 0`, so the answer was `0`, not the median.
3. **`fit` on training data only.** Every leakage rule is a corollary of this.
4. **`Pipeline` makes leakage structurally impossible** — that's the reason it exists, not convenience.
5. **Accuracy lies on imbalanced data.** 73.5% here by predicting nothing.

---

# 4. Async brief

## Released today

**Assessment simulation — due before Day 7.** Self-timed and self-scored. Sit it in one block, not in pieces.

## Practice

**pandas** — on either dataset:
1. Monthly sales trend with a 3-month moving average (`rolling(3).mean()`)
2. Top 3 sub-categories by sales **within each region** (`groupby` + `rank`) — the top-N-per-group pattern from Day 2
3. Customers whose average order value exceeds their segment's average (`transform`)
4. Rebuild the normalisation split, but put `City` on `orders` instead of `customers` — verify no information is lost
5. Find the 32 Product IDs with conflicting names and decide which name wins

**PySpark (DE)** — redo exercises 1–3 in Spark. Then run `.explain()` on each and identify the shuffle.

**sklearn (DS)** — swap `LogisticRegression` for `RandomForestClassifier` inside the same Pipeline (change one line). Compare ROC-AUC and feature importances. Then answer: why does the tree not need `StandardScaler`?

**Everyone:** the auto-graded platforms. HackerRank pandas track, LeetCode for DSA.

## Coming up

**Day 5 — Pipelines, Data Quality & Cloud.** Everything today feeds it:

| Today | Day 5 |
|---|---|
| Date parsing failures, blank `TotalCharges` | **validation gates** |
| 32 conflicting Product Names | **hard stop vs quarantine** |
| The normalisation split | **Bronze → Silver → Gold** |
| `Pipeline` | **the ML pipeline: raw → features → train → deploy → monitor** |
| Parquet over CSV | **storage layer design** |

---

## One thing to take from today

Every defect in both datasets was found by **`.info()` and looking at the rows** — not by clever code.

The date format, the float postal code, the text `TotalCharges`, the 11 zero-tenure customers, the 32 conflicting product names. Five real bugs, all found by reading output carefully before writing anything.

**That habit is the job.** The syntax you can look up.